**Support Vector Machines**

_This notebook contains all the sample code and solutions to the exercises in chapter 5._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-ml3/blob/main/05_support_vector_machines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-ml3/blob/main/05_support_vector_machines.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.7 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 7)

It also requires Scikit-Learn ≥ 1.0.1:

In [ ]:
from packaging import version
import sklearn

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

As we did in previous chapters, let's define the default font sizes to make the figures prettier:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

And let's create the `images/svm` folder (if it doesn't already exist), and define the `save_fig()` function which is used through this notebook to save the figures in high-res for the book:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "svm"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

---
# Linear SVM Classification

The book starts with a few figures, before the first code example, so the next three cells generate and save these figures. You can skip them if you want.

## Large Margin Classification

This figure demonstrates the fundamental concept of Support Vector Machines: **finding the optimal decision boundary with maximum margin**.

**Left plot** shows several possible linear decision boundaries (green dashed, magenta solid, and red solid lines) that could separate the two classes (Iris setosa in yellow circles and Iris versicolor in blue squares). While all these lines achieve separation, they are not equally good.

**Right plot** shows the SVM solution: the decision boundary (solid black line) that maximizes the margin between the two classes. The dashed black lines represent the margins, and the highlighted points are the **support vectors** - the critical instances that define the decision boundary. The SVM finds the widest possible "street" between the classes, which typically leads to better generalization on new data.

This example uses a **hard margin** classifier (with very large C=10¹⁰⁰), meaning it does not tolerate any misclassifications and tries to separate the classes perfectly.

In [ ]:
# extra code – this cell generates and saves Figure 5–1

import matplotlib.pyplot as plt
import numpy as np
from sklearn.svm import SVC
from sklearn import datasets

# Load iris dataset and extract petal features
iris = datasets.load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values # two features
y = iris.target # three classes: 0=setosa, 1=versicolor, 2=virginica

# Filter to keep only setosa (0) and versicolor (1) classes
setosa_or_versicolor = (y == 0) | (y == 1)
X = X[setosa_or_versicolor] # two features for the two classes
y = y[setosa_or_versicolor] # binary target variable; 0 or 1

# SVM Classifier model with linear kernel and very large C (hard margin)
svm_clf = SVC(kernel="linear", C=1e100) # large C to approximate hard margin
svm_clf.fit(X, y)

# Bad models - example decision boundaries that are not optimal
x0 = np.linspace(0, 5.5, 200)
pred_1 = 5 * x0 - 20
pred_2 = x0 - 1.8
pred_3 = 0.1 * x0 + 0.5

def plot_svc_decision_boundary(svm_clf, xmin, xmax):
    """Plot the decision boundary and margins of a trained linear SVM classifier.

    Args:
        svm_clf (sklearn.svm.SVC): A trained SVM classifier with a linear kernel.
        xmin (float): The minimum value for the x-axis range.
        xmax (float): The maximum value for the x-axis range.
    """

    w = svm_clf.coef_[0] # weights vector
    b = svm_clf.intercept_[0] # bias term

    # At the decision boundary, w0*x0 + w1*x1 + b = 0
    # => x1 = -w0/w1 * x0 - b/w1
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0] / w[1] * x0 - b / w[1]

    # Calculate margin width and gutter boundaries
    margin = 1/w[1] # margin = 1/||w||, since ||w|| = sqrt(w0^2 + w1^2) and here w1 is used for scaling
    gutter_up = decision_boundary + margin # upper margin
    gutter_down = decision_boundary - margin # lower margin
    svs = svm_clf.support_vectors_ # support vectors

    # Plot decision boundary (solid line) and margins (dashed lines)
    plt.plot(x0, decision_boundary, "k-", linewidth=2, zorder=-2)
    plt.plot(x0, gutter_up, "k--", linewidth=2, zorder=-2)
    plt.plot(x0, gutter_down, "k--", linewidth=2, zorder=-2)
    # Highlight support vectors
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#AAA',
                zorder=-1) # light gray circles

# Create figure with two subplots
fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

# Left subplot: show several possible decision boundaries
plt.sca(axes[0])
plt.plot(x0, pred_1, "g--", linewidth=2)
plt.plot(x0, pred_2, "m-", linewidth=2)
plt.plot(x0, pred_3, "r-", linewidth=2)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", label="Iris setosa")
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper left")
plt.axis((0.0, 5.5, 0.0, 2.0))
plt.gca().set_aspect("equal")
plt.grid()

# Right subplot: show SVM decision boundary with maximum margin
plt.sca(axes[1])
plot_svc_decision_boundary(svm_clf, 0, 5.5)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo")
plt.xlabel("Petal length")
plt.axis((0.0, 5.5, 0.0, 2.0))
plt.gca().set_aspect("equal")
plt.grid()

save_fig("large_margin_classification_plot")
plt.show()

The `plot_svc_decision_boundary()` function visualizes the **decision boundary** and **margins** of a trained linear SVM classifier.

### Inputs:
- **`svm_clf`**: A trained SVM classifier with a linear kernel
- **`xmin`, `xmax`**: The range of x-axis values for plotting

### How it works:

1. **Extract model parameters:**
    - `w = svm_clf.coef_[0]`: The weight vector (coefficients) learned by the SVM
    - `b = svm_clf.intercept_[0]`: The bias term (intercept)

2. **Compute the decision boundary:**
    - The decision boundary is defined by the equation: **w₀·x₀ + w₁·x₁ + b = 0**
    - Solving for x₁: **x₁ = -(w₀/w₁)·x₀ - b/w₁**
    - This gives us a line that separates the two classes

3. **Calculate the margins:**
    - The margin width is **1/||w||**, where ||w|| is the norm of the weight vector
    - For simplicity in this 2D case, it uses `margin = 1/w[1]`
        - This works because the data is aligned such that only w₁ is relevant for the margin calculation; in general, you would compute the full norm
    - **Upper margin (`gutter_up`)**: decision_boundary + margin
        - This is the line parallel to the decision boundary, offset by the margin
    - **Lower margin (`gutter_down`)**: decision_boundary - margin
    - These parallel lines define the "street" between the classes

4. **Identify support vectors:**
    - `svs = svm_clf.support_vectors_`: The training instances that lie on or within the margin

5. **Create the visualization:**
    - **Solid black line**: The decision boundary separating the classes
    - **Dashed black lines**: The margin boundaries (the "gutters")
    - **Highlighted points**: The support vectors that define the decision boundary

This visualization demonstrates how SVMs maximize the margin between classes, showing which training instances (support vectors) are critical for defining the optimal separating hyperplane.

In [ ]:
def plot_svc_decision_boundary_weight_norm(svm_clf, xmin, xmax):
    """Plot the SVM decision boundary and margins, using the L2 norm of the weights."""
    w = svm_clf.coef_[0] # weights vector
    b = svm_clf.intercept_[0] # bias term

    # At the decision boundary, w0*x0 + w1*x1 + b = 0
    # => x1 = -w0/w1 * x0 - b/w1
    x0 = np.linspace(xmin, xmax, 200)
    decision_boundary = -w[0] / w[1] * x0 - b / w[1]

    # Calculate margin width and gutter boundaries
    margin = 1/np.linalg.norm(w) # margin = 1/||w||, here ||w|| = sqrt(w0^2 + w1^2)
    gutter_up = decision_boundary + margin # upper margin
    gutter_down = decision_boundary - margin # lower margin
    svs = svm_clf.support_vectors_ # support vectors

    # Plot decision boundary (solid line) and margins (dashed lines)
    plt.plot(x0, decision_boundary, "k-", linewidth=2, zorder=-2)
    plt.plot(x0, gutter_up, "k--", linewidth=2, zorder=-2)
    plt.plot(x0, gutter_down, "k--", linewidth=2, zorder=-2)
    # Highlight support vectors
    plt.scatter(svs[:, 0], svs[:, 1], s=180, facecolors='#AAA',
                zorder=-1)

# Create figure with two subplots
fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

# Left subplot: show several possible decision boundaries
plt.sca(axes[0])
plt.plot(x0, pred_1, "g--", linewidth=2)
plt.plot(x0, pred_2, "m-", linewidth=2)
plt.plot(x0, pred_3, "r-", linewidth=2)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", label="Iris setosa")
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper left")
plt.axis((0, 5.5, 0, 2))
plt.gca().set_aspect("equal")
plt.grid()

# Right subplot: show SVM decision boundary with maximum margin
plt.sca(axes[1])
plot_svc_decision_boundary(svm_clf, 0, 5.5)
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo")
plt.xlabel("Petal length")
plt.axis((0, 5.5, 0, 2))
plt.gca().set_aspect("equal")
plt.grid()

save_fig("large_margin_classification_plot")
plt.show()

This code snippet demonstrates the **mathematically rigorous** approach to calculating the margin width and gutter boundaries for a linear Support Vector Machine classifier, in contrast to the simplified version you saw earlier.

The key difference here is in how the margin is calculated. This version uses the complete Euclidean norm formula: `margin = 1/np.linalg.norm(w) = 1/np.sqrt(w[0]**2 + w[1]**2)`. This correctly implements the mathematical definition of the SVM margin as $\frac{1}{\|w\|}$, where $\|w\| = \sqrt{w_0^2 + w_1^2}$ is the L2 norm (Euclidean length) of the weight vector.

This is the proper, general-purpose way to compute the margin that works correctly regardless of how the data is oriented or which features dominate. Unlike the simplified `margin = 1/w[1]` approach in the earlier code, this calculation doesn't make any assumptions about the data alignment. The norm captures the true "length" of the weight vector in the feature space, which determines how wide the margin (the "street") can be between the two classes.

The subsequent calculations for `gutter_up` and `gutter_down` remain the same as before—they define the upper and lower boundaries of the margin by adding and subtracting the margin width from the decision boundary.

The variable `svs` extracts the support vectors, which are the training instances that lie on or within these margin boundaries. These support vectors are the critical points that define the optimal decision boundary, and visualizing them helps illustrate why SVMs are called "support vector" machines—the decision boundary is entirely determined by these supporting instances.

**Hard Margin Linear SVM Classification**

A hard margin SVM is a linear classifier that seeks to perfectly separate two classes with the widest possible margin, assuming the data is linearly separable and contains no outliers.

- **Objective:** Find the hyperplane (decision boundary) that maximizes the margin—the distance between the closest points of each class (the support vectors) and the hyperplane.
- **Constraints:** All training points must be correctly classified and lie outside the margin boundaries. No misclassifications or margin violations are allowed.
- **Mathematical Formulation:**
    - Minimize: $\frac{1}{2} \|w\|^2$
    - Subject to: $t_i (w^\top x_i + b) \geq 1$ for all $i$
        - Where $t_i$ is the class label (+1 or -1), $x_i$ are the training instances, $w$ is the weight vector, and $b$ is the bias term.
- **Result:** The SVM finds the optimal separating hyperplane, defined entirely by the support vectors. Points not on the margin do not affect the solution.

**Limitations:** Hard margin SVMs are sensitive to outliers and cannot handle non-separable data. In practice, soft margin SVMs (which allow some violations) are preferred for real-world datasets.

---

## Feature Scaling Importance for SVMs
This figure illustrates the critical importance of feature scaling when using Support Vector Machines (SVMs). SVMs are sensitive to the scale of the input features because they rely on distance calculations to find the optimal decision boundary.

In [ ]:
# extra code – this cell generates and saves Figure 5–2

from sklearn.preprocessing import StandardScaler

Xs = np.array([[1, 50], [5, 20], [3, 80], [5, 60]]).astype(np.float64)
ys = np.array([0, 0, 1, 1])
svm_clf = SVC(kernel="linear", C=100).fit(Xs, ys) # without feature scaling

scaler = StandardScaler()
X_scaled = scaler.fit_transform(Xs) # feature scaling
svm_clf_scaled = SVC(kernel="linear", C=100).fit(X_scaled, ys) # with feature scaling

plt.figure(figsize=(9, 2.7))
plt.subplot(121)
plt.plot(Xs[:, 0][ys==1], Xs[:, 1][ys==1], "bo")
plt.plot(Xs[:, 0][ys==0], Xs[:, 1][ys==0], "ms")
plot_svc_decision_boundary(svm_clf, 0, 6)
plt.xlabel("$x_0$")
plt.ylabel("$x_1$    ", rotation=0)
plt.title("Unscaled")
plt.axis((0, 6, 0, 90))
plt.grid()

plt.subplot(122)
plt.plot(X_scaled[:, 0][ys==1], X_scaled[:, 1][ys==1], "bo")
plt.plot(X_scaled[:, 0][ys==0], X_scaled[:, 1][ys==0], "ms")
scaled_X_min = np.floor(X_scaled[:, 0].min() - 1)
scaled_X_max = np.ceil(X_scaled[:, 0].max() + 1)
plot_svc_decision_boundary(svm_clf_scaled, scaled_X_min, scaled_X_max)
plt.xlabel("$x'_0$")
plt.ylabel("$x'_1$  ", rotation=0)
plt.title("Scaled")
plt.axis((scaled_X_min, scaled_X_max, scaled_X_min, scaled_X_max))
plt.grid()

save_fig("sensitivity_to_feature_scales_plot")
plt.show()

This figure demonstrates the **sensitivity of Support Vector Machines to feature scaling**.

- **SVMs and Feature Scales**: SVMs aim to find a decision boundary that is as far as possible from the nearest instances of each class. If features are not on a similar scale, the SVM will be biased towards the feature with the larger range, potentially leading to a suboptimal decision boundary.

- **Unscaled Data (Left Plot)**:
    - The plot shows a dataset where the vertical feature (`x₁`) has a much larger scale (0-90) than the horizontal feature (`x₀`) (0-6).
    - The resulting decision boundary is almost horizontal, largely ignoring the `x₀` feature. This is because the margin is calculated based on distances, and the large scale of `x₁` dominates this calculation.

- **Scaled Data (Right Plot)**:
    - The same data is plotted after being scaled using Scikit-Learn's `StandardScaler`. Now, both features have a similar scale (mean=0, variance=1).
    - The SVM is now able to find a decision boundary that correctly balances both features, resulting in a much better separation of the classes.

This visualization clearly shows why **it is crucial to scale your data before training an SVM**. Without scaling, the model may perform poorly because it will *unfairly prioritize features with larger value ranges*.

---
## Soft Margin Classification

Soft-margin SVMs allow some violations of the margin (and even misclassifications) to handle noisy or non-separable data, trading a wider margin for a small number of errors.

Primal optimization problem:
- minimize: 1/2 ||w||² + C ∑ ξᵢ
- subject to: tᵢ(wᵀxᵢ + b) ≥ 1 − ξᵢ and ξᵢ ≥ 0 for all i

Equivalent unconstrained objective (hinge loss):
- minimize: 1/2 ||w||² + C ∑ max(0, 1 − tᵢ(wᵀxᵢ + b))

Key points:
- ξᵢ are slack variables measuring how much each sample violates the margin (ξᵢ = 0 on/behind correct side of the margin; 0 < ξᵢ < 1 on the margin; ξᵢ ≥ 1 misclassified).
- C controls the trade-off:
    - Large C: narrow margin, few violations, can overfit and be sensitive to outliers.
    - Small C: wider margin, more violations tolerated, better generalization on noisy data.
- Support vectors are points on the margin or violating it (ξᵢ > 0).
- Always scale features; tune C via cross-validation.

In [ ]:
# extra code – this cell generates and saves Figure 5–3

X_outliers = np.array([[3.4, 1.3], [3.2, 0.8]])
y_outliers = np.array([0, 0])
Xo1 = np.concatenate([X, X_outliers[:1]], axis=0)
yo1 = np.concatenate([y, y_outliers[:1]], axis=0)
Xo2 = np.concatenate([X, X_outliers[1:]], axis=0)
yo2 = np.concatenate([y, y_outliers[1:]], axis=0)

svm_clf2 = SVC(kernel="linear", C=10**9)
svm_clf2.fit(Xo2, yo2)

fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

plt.sca(axes[0])
plt.plot(Xo1[:, 0][yo1==1], Xo1[:, 1][yo1==1], "bs")
plt.plot(Xo1[:, 0][yo1==0], Xo1[:, 1][yo1==0], "yo")
plt.text(0.3, 1.0, "Impossible!", color="red", fontsize=18)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.annotate(
    "Outlier",
    xy=(X_outliers[0][0], X_outliers[0][1]),
    xytext=(2.5, 1.7),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
)
plt.axis((0, 5.5, 0, 2))
plt.grid()

plt.sca(axes[1])
plt.plot(Xo2[:, 0][yo2==1], Xo2[:, 1][yo2==1], "bs")
plt.plot(Xo2[:, 0][yo2==0], Xo2[:, 1][yo2==0], "yo")
plot_svc_decision_boundary(svm_clf2, 0, 5.5)
plt.xlabel("Petal length")
plt.annotate(
    "Outlier",
    xy=(X_outliers[1][0], X_outliers[1][1]),
    xytext=(3.2, 0.08),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
)
plt.axis((0, 5.5, 0, 2))
plt.grid()

save_fig("sensitivity_to_outliers_plot")
plt.show()

This figure illustrates the limitations of hard-margin SVM classification, particularly its sensitivity to outliers and its inability to handle data that is not perfectly linearly separable.

-   **Left Plot (Impossible Separation)**:
    -   This plot shows the Iris dataset with an added outlier that makes the two classes non-linearly separable.
    -   A hard-margin SVM requires that all instances be correctly classified with no margin violations.
    -   Because of the outlier's position, it is impossible to draw a straight line that separates the two classes. Therefore, a hard-margin classifier cannot find a solution for this dataset.

-   **Right Plot (Sensitivity to Outliers)**:
    -   This plot shows the same dataset but with a different outlier. The data is still linearly separable, but the outlier is very close to the decision boundary.
    -   A hard-margin SVM (approximated here by using a very large `C` value) is forced to find a decision boundary that correctly classifies this single outlier.
    -   As a result, the decision boundary is dramatically skewed and has a much smaller margin compared to the boundary without the outlier (as seen in Figure 5-1). This new model is unlikely to generalize well to new data.

These two examples demonstrate why hard-margin classification is often impractical for real-world datasets, which are rarely perfectly clean and separable. This motivates the need for a more flexible model, such as a **soft-margin SVM**, which allows for some margin violations.

---

## Regularization in Linear SVMs

The following code loads the Iris dataset, scales the features, and then trains a linear SVM model (using Scikit-Learn's `LinearSVC` class with `C=1` and the squared hinge loss) to detect Iris virginica flowers.

Note: the default value for the `dual` hyperparameter of the `LinearSVC` and `LinearSVR` estimators will change from `True` to `"auto"` in Scikit-Learn 1.4, so I set `dual=True` throughout this notebook to ensure the output of this notebook remains unchanged.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

iris = load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 2)  # Iris virginica

svm_clf = make_pipeline(StandardScaler(),
                        LinearSVC(C=1, dual=True, random_state=42))
svm_clf.fit(X, y)

Now that the SVM classifier is trained, you can use it to make predictions. For example, on a couple of new flower instances:

In [ ]:
X_new = [[5.5, 1.7], [5.0, 1.5]]
svm_clf.predict(X_new)

The `decision_function()` method computes the signed distance of the input samples to the decision boundary. This score is what the SVM uses to make a prediction.

-   **Sign of the score**: Determines the predicted class. By default, a positive score indicates that the sample belongs to the positive class (in this case, `True` for Iris virginica), while a negative score indicates the negative class.
-   **Magnitude of the score**: Represents the distance from the decision boundary. A larger absolute value means the sample is further from the boundary, and the model is more "confident" in its prediction.

The `predict()` method simply checks the sign of this score to return a final class label. For the two new instances in `X_new`, this method will return two scores, showing how far each point lies from the separating hyperplane.

In [ ]:
svm_clf.decision_function(X_new)

In [ ]:
list(svm_clf)

In [ ]:
scaler = svm_clf[0]
linear_svc = svm_clf[1]

print(f'scaler.mean_ = {scaler.mean_}')
print(f'scaler.scale_ = {scaler.scale_}')

print(f'linear_svc.coef_ = {linear_svc.coef_}')
list(svm_clf)
print(f'linear_svc.intercept_ = {linear_svc.intercept_}')
print(f'linear_svc.decision_function(X_new) = {linear_svc.decision_function(scaler.transform(X_new))}')

---

The `C` hyperparameter controls the trade-off between achieving a wider margin and minimizing training errors (margin violations).

-   **Small `C` (e.g., C=1)**: High regularization → wider margin, more violations tolerated → better generalization but possible underfitting
-   **Large `C` (e.g., C=100)**: Low regularization → narrower margin, fewer violations → tighter fit to training data but risk of overfitting

The following code trains two `LinearSVC` models with different `C` values and visualizes their decision boundaries to illustrate this trade-off.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Create the pipelines
# We don't need to instantiate scaler separately here because the pipeline will clone it anyway.
# We will access the fitted scaler directly from the pipeline later.
svm_clf1 = LinearSVC(C=1, max_iter=10_000, dual=True, random_state=42)
svm_clf2 = LinearSVC(C=100, max_iter=10_000, dual=True, random_state=42)

scaled_svm_clf1 = make_pipeline(StandardScaler(), svm_clf1)
scaled_svm_clf2 = make_pipeline(StandardScaler(), svm_clf2)

scaled_svm_clf1.fit(X, y)
scaled_svm_clf2.fit(X, y)

# 2. Helper function to unscale parameters and inject support vectors
def unscale_and_inject_attributes(pipeline, X, y):
    """
    Unscale the parameters of a scaled SVM pipeline and inject support vectors.

    This function takes a pipeline containing a StandardScaler and a LinearSVC,
    calculates the unscaled weight vector and bias, and injects them back into
    the LinearSVC model. It also manually calculates and injects the support
    vectors, which are not stored by LinearSVC by default.

    Args:
        pipeline (Pipeline): A scikit-learn pipeline containing a StandardScaler
            and a LinearSVC model.
        X (numpy.ndarray): The training data used to fit the pipeline.
        y (numpy.ndarray): The target labels used to fit the pipeline.
    """
    # Extract fitted steps
    scaler = pipeline[0]
    clf = pipeline[1]

    # Calculate unscaled parameters
    # We use the scaler INSIDE the pipeline because that is the one that was fitted
    b = clf.decision_function([-scaler.mean_ / scaler.scale_])
    w = clf.coef_[0] / scaler.scale_

    # Inject back into the classifier (Monkey-patching for visualization)
    clf.intercept_ = np.array([b])
    clf.coef_ = np.array([w])

    # Find support vectors (LinearSVC does not do this automatically)
    # We calculate them manually to allow plot_svc_decision_boundary to work
    t = y * 2 - 1 # convert to +1/-1
    support_vectors_idx = (t * (X.dot(w) + b) < 1).ravel()

    # type: ignore suppresses the linter error because we are adding a new attribute dynamically
    clf.support_vectors_ = X[support_vectors_idx] # type: ignore

# Apply the fix to both models
unscale_and_inject_attributes(scaled_svm_clf1, X, y)
unscale_and_inject_attributes(scaled_svm_clf2, X, y)

# 3. Plotting
fig, axes = plt.subplots(ncols=2, figsize=(10, 2.7), sharey=True)

plt.sca(axes[0])
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^", label="Iris virginica")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs", label="Iris versicolor")
# We pass the classifier instance (svm_clf1) which now has the injected attributes
plot_svc_decision_boundary(svm_clf1, 4, 5.9)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper left")
plt.title(f"$C = {svm_clf1.C}$")
plt.axis((4, 5.9, 0.8, 2.8))
plt.grid()

plt.sca(axes[1])
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
plot_svc_decision_boundary(svm_clf2, 4, 5.99)
plt.xlabel("Petal length")
plt.title(f"$C = {svm_clf2.C}$")
plt.axis((4, 5.9, 0.8, 2.8))
plt.grid()

save_fig("regularization_plot")
plt.show()

**The Effect of the C Hyperparameter**

This code demonstrates how the **C hyperparameter** controls the regularization strength in Support Vector Machines, which determines the trade-off between achieving a wider margin and allowing margin violations.

### What the code does:

1. **Creates the dataset**: Uses Iris virginica (class 2) vs. all other classes as a binary classification problem
2. **Trains two SVM models** with different C values:
    - `svm_clf1`: C=1 (more regularization)
    - `svm_clf2`: C=100 (less regularization)
3. **Converts scaled parameters back to original scale** for visualization purposes
4. **Identifies support vectors** manually (since `LinearSVC` doesn't store them automatically)
5. **Visualizes both models side-by-side**

### Understanding the C parameter:

- **Large C (C=100, right plot)**:
  - Less regularization, narrower margin
  - Model tries harder to correctly classify all training instances
  - More sensitive to individual data points
  - Risk of overfitting
  
- **Small C (C=1, left plot)**:
  - More regularization, wider margin
  - Model tolerates more margin violations
  - More robust to outliers
  - Better generalization on noisy data

### Key observations in the figures:

- The **left plot (C=1)** shows a wider "street" (margin) with potentially more support vectors
- The **right plot (C=100)** shows a narrower margin that fits the training data more tightly
- Support vectors (highlighted points) are instances that lie on or violate the margin boundaries

This visualization demonstrates that choosing the right C value is crucial: too small and you might underfit; too large and you might overfit. Cross-validation should be used to find the optimal value.

#### Unscaling Parameters for Visualization

The `LinearSVC` models were trained on scaled data, so the learned parameters ($w_{scaled}$ and $b_{scaled}$) correspond to the scaled feature space. To visualize the decision boundary on the original data, we need to transform these parameters back to the original space.

The `unscale_and_inject_attributes()` function performs this transformation and manually identifies support vectors (since `LinearSVC` does not store them).

##### 1. Mathematical Derivation for Unscaling

Let the scaling transformation be $z = \frac{x - \mu}{\sigma}$, where $\mu$ is the mean and $\sigma$ is the standard deviation.
The decision function learned on scaled data is:
$$ f(z) = w_{scaled} \cdot z + b_{scaled} $$

Substituting $z$ back into the equation:
$$ f(x) = w_{scaled} \cdot \left( \frac{x - \mu}{\sigma} \right) + b_{scaled} $$
$$ f(x) = \left( \frac{w_{scaled}}{\sigma} \right) \cdot x + \left( b_{scaled} - \frac{w_{scaled} \cdot \mu}{\sigma} \right) $$

Matching this to the unscaled decision function $f(x) = w_{original} \cdot x + b_{original}$, we derive the formulas used in the code:

*   **Unscaled Weights**: $w_{original} = \frac{w_{scaled}}{\sigma}$
    *   *Code:* `w = clf.coef_[0] / scaler.scale_`
*   **Unscaled Bias**: $b_{original} = b_{scaled} - \frac{w_{scaled} \cdot \mu}{\sigma} = w_{scaled} \cdot \left(- \frac{\mu}{\sigma}\right) + b_{scaled}$
    *   *Code:* `b = clf.decision_function([-scaler.mean_ / scaler.scale_])` (This calculates the bias term by evaluating the scaled decision function at $z = -\mu/\sigma$).

##### 2. Identifying Support Vectors

Unlike `SVC`, `LinearSVC` minimizes the squared hinge loss and does not automatically track support vectors. We identify them manually by finding instances that violate the "safe" margin condition.

For a target label $t_i \in \{-1, +1\}$, a point is correctly classified and safely outside the margin if $t_i (w \cdot x_i + b) \ge 1$.
Any point violating this is considered a **support vector**:

$$ t_i (w \cdot x_i + b) < 1 $$

This condition captures points that are:
1.  **On the margin boundary** (value $\approx 1$)
2.  **Inside the margin** ($0 \le \text{value} < 1$)
3.  **Misclassified** ($\text{value} < 0$)


---

# Nonlinear SVM Classification

Real-world datasets are rarely linearly separable. For nonlinear datasets, one approach is to engineer new features (such as polynomial features) and then apply a linear SVM on the transformed feature space.

However, SVMs offer a more elegant solution: the **kernel trick**. This mathematical technique allows SVMs to fit nonlinear decision boundaries *without explicitly transforming the input features*, making it both computationally efficient and powerful for handling complex, nonlinear patterns in data.

In [ ]:
# extra code – this cell generates and saves Figure 5–5

X1D = np.linspace(-4, 4, 9).reshape(-1, 1)
X2D = np.c_[X1D, X1D**2]
y = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0])

plt.figure(figsize=(10, 3))

plt.subplot(121)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.plot(X1D[:, 0][y==0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][y==1], np.zeros(5), "g^")
plt.yticks([])
plt.xlabel("$x_1$")
plt.axis((-4.5, 4.5, -0.2, 0.2))

plt.subplot(122)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(X2D[:, 0][y==0], X2D[:, 1][y==0], "bs")
plt.plot(X2D[:, 0][y==1], X2D[:, 1][y==1], "g^")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$  ", rotation=0)
plt.yticks([0, 4, 8, 12, 16])
plt.plot([-4.5, 4.5], [6.5, 6.5], "r--", linewidth=3)
plt.axis((-4.5, 4.5, -1, 17))

plt.subplots_adjust(right=1)

save_fig("higher_dimensions_plot", tight_layout=False)
plt.show()

This figure illustrates how adding features can transform a non-linearly separable dataset into a linearly separable one, a key concept for understanding how SVMs handle non-linear data.

- **Left Plot (Original 1D Data)**:
    - This plot shows a simple one-dimensional dataset where the two classes (blue squares and green triangles) are mixed.
    - It is impossible to separate these classes with a single point (which would be the 1D equivalent of a line). The data is **not linearly separable**.

- **Right Plot (Transformed 2D Data)**:
    - To solve this, we add a new feature: `x₂ = x₁²`. The data is now represented in a two-dimensional space.
    - The horizontal axis is the original feature `x₁`, and the vertical axis is the new squared feature `x₂`.
    - In this new, higher-dimensional feature space, the dataset becomes **linearly separable**.
    - The red dashed line represents a simple linear decision boundary that can now perfectly separate the two classes.

This example demonstrates the principle behind using techniques like `PolynomialFeatures`: by projecting the data into a higher-dimensional space, a complex, non-linear problem can be converted into a simpler, linear one that a linear classifier can solve.

---
Then, the following code implements a polynomial SVM by creating a `Pipeline` that combines three steps:

1.  **`PolynomialFeatures`**: This transformer adds polynomial combinations of the features to the dataset. In this case, it creates 3rd-degree polynomial features, effectively mapping the original 2D data into a higher-dimensional space where it is more likely to be linearly separable.
2.  **`StandardScaler`**: This step scales the features to have zero mean and unit variance. Feature scaling is crucial for SVMs, as they are sensitive to the magnitude of features.
3.  **`LinearSVC`**: A linear SVM classifier is then trained on the transformed and scaled data.

By chaining these components, the pipeline can learn a non-linear decision boundary for the `moons` dataset.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import PolynomialFeatures

X, y = make_moons(n_samples=100, noise=0.15, random_state=42)

polynomial_svm_clf = make_pipeline(
    PolynomialFeatures(degree=3),
    StandardScaler(),
    LinearSVC(C=10, max_iter=10_000, dual=True, random_state=42)
)
polynomial_svm_clf.fit(X, y)

In [ ]:
# extra code – this cell generates and saves Figure 5–6

def plot_dataset(X, y, axes):
    """
    Plots the dataset with different markers for each class.

    Args:
        X (numpy.ndarray): The feature matrix.
        y (numpy.ndarray): The target labels.
        axes (list): The axis limits [xmin, xmax, ymin, ymax].
    """
    plt.plot(X[:, 0][y==0], X[:, 1][y==0], "bs")
    plt.plot(X[:, 0][y==1], X[:, 1][y==1], "g^")
    plt.axis(axes)
    plt.grid(True)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$", rotation=0)

def plot_predictions(clf, axes):
    """
    Plots the decision boundary and decision function of a classifier.

    Args:
        clf: The trained classifier (must have predict and decision_function methods).
        axes (list): The axis limits [xmin, xmax, ymin, ymax].
    """
    # Create a meshgrid covering the specified axes range
    x0s = np.linspace(axes[0], axes[1], 100)
    x1s = np.linspace(axes[2], axes[3], 100)
    x0, x1 = np.meshgrid(x0s, x1s)

    # Flatten the meshgrid to create a feature matrix for prediction
    X = np.c_[x0.ravel(), x1.ravel()]

    # Make predictions and compute decision function values
    y_pred = clf.predict(X).reshape(x0.shape)
    y_decision = clf.decision_function(X).reshape(x0.shape)

    # Plot the decision regions and the decision function contours
    plt.contourf(x0, x1, y_pred, cmap="brg", alpha=0.2)
    plt.contourf(x0, x1, y_decision, cmap="brg", alpha=0.1)

plot_predictions(polynomial_svm_clf, [-1.5, 2.5, -1, 1.5]) # Plot the decision boundary and decision function
plot_dataset(X, y, [-1.5, 2.5, -1, 1.5]) # Plot the dataset on the same axes

save_fig("moons_polynomial_svc_plot")
plt.show()

This code visualizes the decision boundary learned by the polynomial SVM classifier on the `moons` dataset.

### How it works:

1.  **`plot_dataset(X, y, axes)`**: A helper function that creates a scatter plot of the dataset, showing the two classes with different markers (blue squares and green triangles).
2.  **`plot_predictions(clf, axes)`**: This function visualizes the classifier's decision boundary.
    - It creates a grid of points covering the entire plot area.
    - For each point, it predicts the class and calculates the decision function score.
        - `y_pred` contains the predicted class labels for each point in the grid.
        - `y_decision` contains the decision function values for each point, which indicate the distance from the decision boundary.
    - It then uses `contourf` to draw filled contour plots, showing the decision regions (colored areas) and the decision function values (subtler color gradients indicating the margin).
3.  **Plotting**:
    - `plot_predictions()` is called to draw the decision boundary of the `polynomial_svm_clf`.
    - `plot_dataset()` is called to overlay the original data points on the plot.

The resulting figure shows how the pipeline, by adding polynomial features, is able to learn a complex, non-linear boundary that effectively separates the two classes of the `moons` dataset.

---
## Polynomial Kernel
Adding polynomial features is a simple way to handle non-linear datasets. However, at a low polynomial degree, this method may not be able to handle very complex datasets, and with a high polynomial degree, it can create a huge number of features, making the model too slow.

Fortunately, when using SVMs you can apply a powerful mathematical technique called the **kernel trick**. It delivers the same result as if you had added many polynomial features, even with a very high degree, without actually having to add them. This *avoids the combinatorial explosion of features*, making the process much more efficient.

The kernel trick is implemented by the `SVC` class. The following code trains an SVM classifier using a 3rd-degree polynomial kernel. The `degree` hyperparameter sets the polynomial degree, and `coef0` controls how much the model is influenced by high-degree versus low-degree polynomials.

In [ ]:
from sklearn.svm import SVC

poly_kernel_svm_clf = make_pipeline(StandardScaler(),
                                    SVC(kernel="poly", degree=3, coef0=1, C=5))
poly_kernel_svm_clf.fit(X, y)

### Using **polynomial kernel trick** with SVMs

Instead of manually adding polynomial features, this approach uses the `SVC` class with `kernel="poly"` to achieve the same result more efficiently.

Here's a breakdown of the code:

1.  **`make_pipeline(...)`**: Creates a processing pipeline that chains multiple steps together.
    *   **`StandardScaler()`**: The first step scales the features. This is crucial for SVMs to ensure that all features contribute equally to the decision boundary.
    *   **`SVC(kernel="poly", degree=3, coef0=1, C=5)`**: The second step is the SVM classifier itself, configured with:
        *   `kernel="poly"`: Specifies the use of the polynomial kernel.
        *   `degree=3`: Sets the polynomial degree to 3. The model will find a decision boundary equivalent to one in a feature space expanded with 3rd-degree polynomial features.
        *   `coef0=1`: This hyperparameter controls the influence of high-degree vs. low-degree terms in the polynomial.
            - *A higher `coef0` value increases the influence of higher-degree polynomials*, leading to more complex decision boundaries.
        *   `C=5`: The regularization parameter. It balances the trade-off between a wider margin and minimizing classification errors.

2.  **`.fit(X, y)`**: The pipeline is trained on the dataset `X` and `y`. The data is automatically passed through the scaler before being fed to the SVM classifier.

This approach is a more powerful and computationally efficient way to train a non-linear SVM compared to manually creating polynomial features with `PolynomialFeatures`.

In [ ]:
# extra code – this cell generates and saves Figure 5–7

# Train another SVM with a polynomial kernel of degree 10 and coef0 of 100
poly100_kernel_svm_clf = make_pipeline(
    StandardScaler(),
    SVC(kernel="poly", degree=10, coef0=100, C=5)
)
poly100_kernel_svm_clf.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(10.5, 4), sharey=True)

plt.sca(axes[0])
plot_predictions(poly_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.4, -1, 1.5])
plt.title("degree=3, coef0=1, C=5")

plt.sca(axes[1])
plot_predictions(poly100_kernel_svm_clf, [-1.5, 2.45, -1, 1.5])
plot_dataset(X, y, [-1.5, 2.4, -1, 1.5])
plt.title("degree=10, coef0=100, C=5")
plt.ylabel("")

save_fig("moons_kernelized_polynomial_svc_plot")
plt.show()

we train another polynomial kernel SVM, this time with a high degree (10) and a large `coef0` value (100). It then plots the decision boundaries of both models.

- **Left Plot**: Shows the 3rd-degree polynomial kernel SVM. It provides a reasonable, non-linear boundary for the `moons` dataset.
- **Right Plot**: Shows the 10th-degree polynomial kernel SVM. This model is much more complex and produces a highly irregular decision boundary, which is a clear sign of **overfitting**.

This visualization demonstrates the trade-off when choosing the `degree` hyperparameter:
- A **low degree** might lead to underfitting (the model is too simple).
- A **high degree** can lead to overfitting (the model is too complex and captures noise in the training data).

The `coef0` hyperparameter also influences the model's complexity, controlling how much the model is affected by high-degree versus low-degree polynomials. Finding the right balance for these hyperparameters is crucial for good generalization.

---
## Similarity Features

Another approach to solving nonlinear problems is to add features computed using a **similarity function**, which measures how much each instance resembles a particular *landmark*. For example, let's take the 1D dataset from earlier and add two landmarks at $x_1 = -2$ and $x_1 = 1$.

We define the similarity function as the **Gaussian Radial Basis Function (RBF)** with $\gamma = 0.3$:

$\phi_{\gamma}(\mathbf{x}, \boldsymbol{\ell}) = \exp(-\gamma \|\mathbf{x} - \boldsymbol{\ell}\|^2)$

where $\boldsymbol{\ell}$ is the landmark, and $\|\mathbf{x} - \boldsymbol{\ell}\|^2$ is the squared Euclidean distance between the instance $\mathbf{x}$ and the landmark $\boldsymbol{\ell}$.

This bell-shaped function ranges from 0 (for instances far from the landmark) to 1 (for instances at the landmark). The parameter $\gamma$ controls the width of the bell: a smaller $\gamma$ results in a wider bell, while a larger $\gamma$ creates a narrower bell.

Let's compute the new features for the instance $x_1 = -1$. It is located at a distance of 1 from the first landmark and 2 from the second. Its new features are:
- $x_2 = \exp(-0.3 \times 1^2) \approx 0.74$
- $x_3 = \exp(-0.3 \times 2^2) \approx 0.30$

As a result, the instance $x_1 = -1$ is now represented by the feature vector $[x_1, x_2, x_3] \approx [-1, 0.74, 0.30]$ in a 3D feature space.

The figure below shows the original 1D dataset on the left and the transformed 2D dataset on the right. As you can see, the transformed dataset is now linearly separable.

In [ ]:
# extra code – this cell generates and saves Figure 5–8

def gaussian_rbf(x, landmark, gamma):
    return np.exp(-gamma * np.linalg.norm(x - landmark, axis=1)**2)

gamma = 0.3

x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x2s = gaussian_rbf(x1s, -2, gamma) # similarity to landmark x2 = -2
x3s = gaussian_rbf(x1s, 1, gamma) # similarity to landmark x3 = 1

XK = np.c_[gaussian_rbf(X1D, -2, gamma), gaussian_rbf(X1D, 1, gamma)] # transformed dataset
yk = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0]) # target labels

plt.figure(figsize=(10.5, 4))

plt.subplot(121)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.scatter(x=[-2, 1], y=[0, 0], s=150, alpha=0.5, c="red")
plt.plot(X1D[:, 0][yk==0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][yk==1], np.zeros(5), "g^")
plt.plot(x1s, x2s, "g--")
plt.plot(x1s, x3s, "b:")
plt.gca().get_yaxis().set_ticks([0, 0.25, 0.5, 0.75, 1])
plt.xlabel("$x_1$")
plt.ylabel("Similarity")
plt.annotate(
    r'$\mathbf{x}$',
    xy=(X1D[3, 0], 0),
    xytext=(-0.5, 0.20),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
    fontsize=16,
)
plt.text(-2, 0.9, "$x_2$", ha="center", fontsize=15)
plt.text(1, 0.9, "$x_3$", ha="center", fontsize=15)
plt.axis((-4.5, 4.5, -0.1, 1.1))

plt.subplot(122)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(XK[:, 0][yk==0], XK[:, 1][yk==0], "bs")
plt.plot(XK[:, 0][yk==1], XK[:, 1][yk==1], "g^")
plt.xlabel("$x_2$")
plt.ylabel("$x_3$  ", rotation=0)
plt.annotate(
    r'$\phi\left(\mathbf{x}\right)$',
    xy=(XK[3, 0], XK[3, 1]),
    xytext=(0.65, 0.50),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
    fontsize=16,
)
plt.plot([-0.1, 1.1], [0.57, -0.1], "r--", linewidth=3)
plt.axis((-0.1, 1.1, -0.1, 1.1))

plt.subplots_adjust(right=1)

save_fig("kernel_method_plot")
plt.show()

This figure shows how adding **similarity features** can make a non-linearly separable dataset linearly separable. It uses the Gaussian Radial Basis Function (RBF) as the similarity function.

### How the code works:

1.  **`gaussian_rbf(x, landmark, gamma)` function**:
    *   This function calculates the similarity between a data point `x` and a chosen `landmark`.
    *   It uses the RBF formula: `exp(-gamma * ||x - landmark||²)`
    *   The result is a value between 0 (far from the landmark) and 1 (at the landmark).
    *   `gamma` controls the width of the similarity "bell curve".

2.  **Data Transformation**:
    *   The code starts with the 1D dataset `X1D` (which is not linearly separable).
    *   It defines two landmarks at `x = -2` and `x = 1`.
    *   It then creates a new 2D dataset `XK` by computing two new features for each point in `X1D`:
        *   **New feature 1 (`x₂`)**: The similarity of the point to the landmark at -2.
        *   **New feature 2 (`x₃`)**: The similarity of the point to the landmark at 1.

3.  **Visualization**:
    *   **Left Plot**:
        *   Shows the original 1D data points (blue squares, green triangles) on the horizontal axis.
        *   The two landmarks are shown as red circles.
        *   The two bell-shaped curves (green dashed and blue dotted) represent the RBF similarity functions centered on each landmark. They show how the new feature values (`x₂` and `x₃`) are calculated for any given `x₁`.
    *   **Right Plot**:
        *   Shows the new 2D dataset `XK`, where the axes are the new similarity features (`x₂` and `x₃`).
        *   In this new feature space, the data becomes **linearly separable**, as indicated by the red dashed line which can now perfectly separate the two classes.

This illustrates the core idea behind the RBF kernel in SVMs: by transforming the data based on similarity to landmarks, a linear classifier can solve a non-linear problem.

For comparison, the following example selects different landmarks at $x_1 = -1$ and $x_1 = 1$.

In [ ]:
from numpy import square


x1s = np.linspace(-4.5, 4.5, 200).reshape(-1, 1)
x2s = gaussian_rbf(x1s, -1, gamma) # similarity to landmark x2 = -1
x3s = gaussian_rbf(x1s, 1, gamma) # similarity to landmark x3 = 1

XK = np.c_[gaussian_rbf(X1D, -1, gamma), gaussian_rbf(X1D, 1, gamma)] # transformed dataset
yk = np.array([0, 0, 1, 1, 1, 1, 1, 0, 0]) # target labels

plt.figure(figsize=(10.5, 4))

plt.subplot(121)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.scatter(x=[-1, 1], y=[0, 0], s=150, alpha=0.5, c="red")
plt.plot(X1D[:, 0][yk==0], np.zeros(4), "bs")
plt.plot(X1D[:, 0][yk==1], np.zeros(5), "g^")
plt.plot(x1s, x2s, "g--")
plt.plot(x1s, x3s, "b:")
plt.yticks([0, 0.25, 0.5, 0.75, 1])
plt.xlabel("$x_1$")
plt.ylabel("Similarity")
plt.annotate(
    r'$\mathbf{x}$',
    xy=(X1D[3, 0], 0),
    xytext=(-0.5, 0.20),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
    fontsize=16,
)
plt.text(-1, 0.9, "$x_2$", ha="center", fontsize=15)
plt.text(1, 0.9, "$x_3$", ha="center", fontsize=15)
plt.axis((-4.5, 4.5, -0.1, 1.1))

plt.subplot(122)
plt.grid(True)
plt.axhline(y=0, color='k')
plt.axvline(x=0, color='k')
plt.plot(XK[:, 0][yk==0], XK[:, 1][yk==0], "bs:")
plt.plot(XK[:, 0][yk==1], XK[:, 1][yk==1], "g^:")
plt.xlabel("$x_2$")
plt.ylabel("$x_3$  ", rotation=0)
plt.annotate(
    r'$\phi\left(\mathbf{x}\right)$',
    xy=(XK[3, 0], XK[3, 1]),
    xytext=(0.65, 0.50),
    ha="center",
    arrowprops=dict(facecolor='black', shrink=0.1),
    fontsize=16,
)
plt.plot([-0.1, 1.1], [0.68, -0.68], "r--", linewidth=3)
plt.axis((-0.1, 1.1, -0.1, 1.1))
plt.gca().set_aspect("equal")

plt.subplots_adjust(right=1)

save_fig("kernel_method_plot_2")
plt.show()

This figure shows the transformed dataset using the Gaussian RBF kernel with landmarks at $x_1 = -1$ and $x_1 = 1$.

- **Left Plot**:
    - Shows the original 1D dataset.
    - The two landmarks are indicated by red circles at $x_1 = -1$ and $x_1 = 1$.
    - The green dashed curve shows the similarity to the first landmark ($x_2$).
    - The blue dotted curve shows the similarity to the second landmark ($x_3$).

- **Right Plot**:
    - Shows the dataset transformed into the new 2D feature space ($x_2$ vs $x_3$).
    - In this space, the instances are plotted based on their similarity to the two landmarks.
    - As you can see, the classes are now perfectly linearly separable by the red dashed line.

**Comparison with previous example:**
In the previous example with landmarks at $x_1 = -2$ and $x_1 = 1$, the landmarks were asymmetrically placed, leading to a skewed decision boundary. Here, the landmarks are symmetrically placed around the data clusters (at -1 and 1). This symmetry results in a cleaner transformation where the data forms a U-shaped curve in the new feature space, allowing for a simple linear separation. This demonstrates how choosing landmarks that reflect the data's structure can yield a better feature space.

---
## Gaussian RBF Kernel
Just like the polynomial features method, the similarity features method can be useful with any machine learning algorithm, but it may be computationally expensive to compute all the additional features, especially on large training sets. Once again the kernel trick does its SVM magic: it makes it possible to obtain a similar result as if you had added many similarity features, without actually having to add them.

Let's try the `SVC` class with the Gaussian RBF kernel. The following code trains an `SVC` using a Gaussian RBF kernel.

In [ ]:
rbf_kernel_svm_clf = make_pipeline(StandardScaler(),
                                   SVC(kernel="rbf", gamma=5, C=0.001))
rbf_kernel_svm_clf.fit(X, y)

In [ ]:
# extra code – this cell generates and saves Figure 5–9

from sklearn.svm import SVC

gamma1, gamma2 = 0.1, 5
C1, C2 = 0.001, 1000
hyperparams = (gamma1, C1), (gamma1, C2), (gamma2, C1), (gamma2, C2)

svm_clfs = []
for gamma, C in hyperparams:
    rbf_kernel_svm_clf = make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", gamma=gamma, C=C)
    )
    rbf_kernel_svm_clf.fit(X, y)
    svm_clfs.append(rbf_kernel_svm_clf)

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10.5, 7), sharex=True, sharey=True)

for i, svm_clf in enumerate(svm_clfs):
    plt.sca(axes[i // 2, i % 2])
    plot_predictions(svm_clf, [-1.5, 2.45, -1, 1.5])
    plot_dataset(X, y, [-1.5, 2.45, -1, 1.5])
    gamma, C = hyperparams[i]
    plt.title(f"gamma={gamma}, C={C}")
    if i in (0, 1):
        plt.xlabel("")
    if i in (1, 3):
        plt.ylabel("")

save_fig("moons_rbf_svc_plot")
plt.show()

This code trains and visualizes four different SVM models using the **Gaussian RBF kernel** to demonstrate the effect of the `gamma` and `C` hyperparameters.

### How the code works:

1.  **Hyperparameter Grid**: It defines a grid of `gamma` values (0.1, 5) and `C` values (0.001, 1000).
2.  **Model Training**: It loops through all four combinations of these hyperparameters. For each pair:
    *   It creates a `pipeline` that first scales the data (`StandardScaler`) and then trains an `SVC` with `kernel="rbf"` using the specified `gamma` and `C`.
    *   It fits the model to the `moons` dataset.
3.  **Visualization**: It creates a 2x2 grid of plots, where each subplot shows the decision boundary of one of the trained models.

### Understanding the figures and hyperparameters:

The plots show how `gamma` and `C` control the model's complexity and regularization.

-   **`gamma`**: Controls the shape of the decision boundary.
    -   **Small `gamma`** (top row): Creates a smooth, wide boundary. The model is simpler and may **underfit**.
    -   **Large `gamma`** (bottom row): Creates a complex, irregular boundary that closely follows the training data. The model is more complex and may **overfit**.

-   **`C`**: Controls the trade-off between a wide margin and minimizing classification errors.
    -   **Small `C`** (left column): High regularization. The model prefers a wider margin, even if it means more margin violations. This can lead to **underfitting**.
    -   **Large `C`** (right column): Low regularization. The model tries to classify every point correctly, leading to a narrower margin and a more complex boundary. This can lead to **overfitting**.

**In summary**:
-   Increasing `gamma` or `C` increases the model's complexity and the risk of overfitting.
-   Decreasing `gamma` or `C` increases regularization and the risk of underfitting.
-   The best model is often found by tuning these hyperparameters to achieve a good balance, like the one in the top-right plot (`gamma=0.1`, `C=1000`).

---
# SVM Regression
As we have seen, SVMs are very versatile: not only do they support various kernels, but they can also be used for regression. The trick is to **reverse the objective**: instead of trying to fit the largest possible street between two classes while limiting margin violations, SVM Regression tries to fit as many instances as possible *on* the street while limiting margin violations (i.e., instances *off* the street). The width of the street is controlled by a hyperparameter, `epsilon` (ε).

Adding more training instances within the margin does not affect the model’s predictions; thus, the model is said to be *ϵ-insensitive*.

Just like for SVM classification, SVM Regression can be used to perform both linear and nonlinear regression, thanks to the kernel trick.

In [ ]:
from sklearn.svm import LinearSVR

# extra code – these 3 lines generate a simple linear dataset
np.random.seed(42)
X = 2 * np.random.rand(50, 1) # 50 samples, 1 feature, values between 0 and 2
# Target values with some noise, y = 4 + 3x + noise
y = 4 + 3 * X[:, 0] + np.random.randn(50)

svm_reg = make_pipeline(StandardScaler(),
                        LinearSVR(epsilon=0.5, dual=True, random_state=42))
svm_reg.fit(X, y)

This code trains a **Linear Support Vector Regression (SVR)** model on a simple, randomly generated linear dataset.

### How it works:

1.  **Generate Data**:
    *   A simple linear dataset is created with 50 instances.
    *   `X` is a single feature with random values between 0 and 2.
    *   `y` is calculated based on a linear equation (`y = 4 + 3x`) with added Gaussian noise to simulate real-world data.

2.  **Create a Pipeline**:
    *   `make_pipeline` is used to chain two steps: feature scaling and model training.
    *   **`StandardScaler()`**: This step scales the input features. Scaling is crucial for SVMs to ensure they perform correctly.
    *   **`LinearSVR(...)`**: This is the *linear SVM regression* model.
        *   `epsilon=0.5`: This hyperparameter defines the width of the margin (the "street"). The model is not penalized for prediction errors smaller than this value.
        *   `dual=True`: This is set for compatibility, as the default will change in future Scikit-Learn versions.

3.  **Train the Model**:
    *   `svm_reg.fit(X, y)` trains the entire pipeline. The data is first scaled and then used to train the `LinearSVR` model.

This code trains two `LinearSVR` models with different `epsilon` values and visualizes them to illustrate the effect of this hyperparameter.

### How it works:

1.  **Two Models**:
    *   `svm_reg`: Trained with `epsilon=0.5`.
    *   `svm_reg2`: Trained with `epsilon=1.2`.

2.  **Helper Functions**:
    *   `find_support_vectors()`: Identifies the support vectors—instances where the prediction error is greater than or equal to `epsilon`.
    *   `plot_svm_regression()`: Visualizes the SVR model, including:
        *   The regression line (solid black).
        *   The margin defined by `epsilon` (dashed lines).
        *   The original data points (blue circles).
        *   The support vectors (highlighted gray circles).

3.  **Visualization**:
    *   **Left Plot (`epsilon=0.5`)**: Shows a model with a narrow margin. More data points fall outside this margin and become support vectors, influencing the model's fit.
    *   **Right Plot (`epsilon=1.2`)**: Shows a model with a wider margin. Fewer points are support vectors because more instances fit inside the wider "street," making the model less sensitive to their exact position.

The figures clearly demonstrate that `epsilon` controls the width of the margin in SVM regression. A larger `epsilon` results in a wider "street" and fewer support vectors, leading to a more regularized model.

In [ ]:
# Investigate the model
print(f'1st Step in pipeline: {svm_reg[0]}')  # scaler
print(f'Mean: {svm_reg[0].mean_}, Scale: {svm_reg[0].scale_}  # scaler mean and scale')
print(f'2nd Step in pipeline: {svm_reg[1]}')
print(f'Coefficients: {svm_reg[1].coef_}, Intercept: {svm_reg[1].intercept_})') # linear SVR parameters
print(f'support vectors: {getattr(svm_reg[1], "support_", "Not available in LinearSVR")}') # support vectors, not available in LinearSVR
print(f'{svm_reg[1].__dict__.keys()}')  # all attributes of LinearSVR

In [ ]:
# extra code – this cell generates and saves Figure 5–10

def find_support_vectors(svm_reg, X, y):
    """Finds the indices of support vectors for a trained SVM regression model.

    This function identifies the instances that lie off the margin (i.e., where the
    absolute error is greater than or equal to epsilon). These instances are the
    support vectors in SVM regression.

    Args:
        svm_reg: A trained Support Vector Regression model (e.g., LinearSVR).
            It is expected to be a pipeline or estimator with a `predict` method
            and an `epsilon` attribute (accessible via the last step if it's a pipeline).
        X (numpy.ndarray): The feature matrix used for prediction.
        y (numpy.ndarray): The target values.

    Returns:
        numpy.ndarray: An array of indices corresponding to the support vectors in X.
    """
    y_pred = svm_reg.predict(X)
    epsilon = svm_reg[-1].epsilon
    off_margin = np.abs(y - y_pred) >= epsilon
    return np.argwhere(off_margin)

def plot_svm_regression(svm_reg, X, y, axes):
    """Plots the SVM regression model predictions and the margin.

    This function visualizes the regression line, the epsilon-insensitive margin,
    the original data points, and highlights the support vectors.

    Args:
        svm_reg: A trained Support Vector Regression model (e.g., LinearSVR).
            It is expected to be a pipeline or estimator with a `predict` method
            and an `epsilon` attribute (accessible via the last step if it's a pipeline).
        X (numpy.ndarray): The feature matrix used for training/prediction.
        y (numpy.ndarray): The target values.
        axes (list): The axis limits [xmin, xmax, ymin, ymax].
    """
    # Generate points for prediction line
    x1s = np.linspace(axes[0], axes[1], 100).reshape(100, 1)
    y_pred = svm_reg.predict(x1s)
    epsilon = svm_reg[-1].epsilon
    plt.plot(x1s, y_pred, "k-", linewidth=2, label=r"$\hat{y}$", zorder=-2)
    plt.plot(x1s, y_pred + epsilon, "k--", zorder=-2) # upper epsilon margin
    plt.plot(x1s, y_pred - epsilon, "k--", zorder=-2) # lower epsilon margin

    # Find support vectors
    support_vectors_idx = find_support_vectors(svm_reg, X, y)
    plt.scatter(X[support_vectors_idx], y[support_vectors_idx], s=180,
                facecolors='#AAA', zorder=-1) # highlight support vectors
    plt.plot(X, y, "bo")
    plt.xlabel("$x_1$")
    plt.legend(loc="upper left")
    plt.axis(axes)

svm_reg2 = make_pipeline(StandardScaler(),
                         LinearSVR(epsilon=1.2, dual=True, random_state=42))
svm_reg2.fit(X, y)

eps_x1 = 1
eps_y_pred = svm_reg2.predict([[eps_x1]])

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_reg, X, y, [0, 2, 3, 11])
plt.title(f"epsilon={svm_reg[-1].epsilon}")
plt.ylabel("$y$", rotation=0)
plt.grid()
plt.sca(axes[1])
plot_svm_regression(svm_reg2, X, y, [0, 2, 3, 11])
plt.title(f"epsilon={svm_reg2[-1].epsilon}")
plt.annotate(
        '', xy=(eps_x1, eps_y_pred), xycoords='data',
        xytext=(eps_x1, eps_y_pred - svm_reg2[-1].epsilon),
        textcoords='data', arrowprops={'arrowstyle': '<->', 'linewidth': 1.5}
    )
plt.text(0.90, 5.4, r"$\epsilon$", fontsize=16)
plt.grid()
save_fig("svm_regression_plot")
plt.show()

This code visualizes and compares two Linear Support Vector Regression (SVR) models with different `epsilon` values to illustrate its effect on the regression fit.

### Code Explanation:

1.  **`find_support_vectors(svm_reg, X, y)` function**:
    *   This helper function identifies the support vectors for an SVR model.
    *   In SVR, support vectors are the training instances that lie *outside* the margin (the "street").
    *   It calculates the absolute difference between the true values (`y`) and the model's predictions (`y_pred`).
    *   It returns the indices of the instances where this error is greater than or equal to the model's `epsilon` value.

2.  **`plot_svm_regression(svm_reg, X, y, axes)` function**:
    *   This function visualizes the results of an SVR model.
    *   It plots the model's prediction line (solid black).
    *   It plots the margin boundaries (dashed lines) at a distance of `epsilon` above and below the prediction line.
    *   It highlights the support vectors (instances outside the margin) with large gray circles.
    *   It plots all the original data points as blue circles.

3.  **Main script**:
    *   It trains a second SVR model, `svm_reg2`, with a larger margin (`epsilon=1.2`). The first model, `svm_reg` (with `epsilon=0.5`), was trained in the previous cell.
    *   It uses the `find_support_vectors` function within `plot_svm_regression` to identify and highlight the support vectors for both models.
    *   It creates a figure with two subplots to compare the two models side-by-side.
    *   The left plot shows the model with `epsilon=0.5`.
    *   The right plot shows the model with `epsilon=1.2` and includes an annotation to visually represent the `epsilon` margin.

### Meaning of the Figures:

The two plots demonstrate how the `epsilon` hyperparameter controls the trade-off between model complexity and error tolerance in SVR.

*   **Left Plot (`epsilon=0.5`)**:
    *   With a small `epsilon`, the margin (the "street") is narrow.
    *   The model tries to fit the data tightly, and as a result, more data points fall outside the margin.
    *   These numerous support vectors (highlighted points) pull the regression line towards them, making the model more sensitive to individual data points.

*   **Right Plot (`epsilon=1.2`)**:
    *   With a large `epsilon`, the margin is wide.
    *   The model is "ϵ-insensitive," meaning it does not penalize errors for any points that fall *inside* this wide street.
    *   Fewer points end up outside the margin, so there are fewer support vectors.
    *   The model is less influenced by the exact position of most data points, resulting in a more regularized and potentially more generalizable fit.

In essence, `epsilon` acts as a regularization parameter: increasing `epsilon` leads to a more regularized model with fewer support vectors.

---


## Nonlinear SVM Regression

To handle non-linear regression tasks, you can use an SVR with a non-linear kernel. The following code trains a Support Vector Machine for regression using a 2nd-degree polynomial kernel on a simple quadratic dataset.

The figures below show the result of training two polynomial SVR models on this data. The model on the left has a very small `C` value, leading to high regularization (i.e., a wider margin). The model on the right has a very large `C` value, leading to low regularization (i.e., a narrower margin). As you can see, the highly regularized model on the left underfits the training data, while the less regularized model on the right provides a much better fit. This illustrates how the `C` hyperparameter can be used to control the trade-off between fitting the training data and generalizing to new data.

In [ ]:
from sklearn.svm import SVR

# extra code – these 3 lines generate a simple quadratic dataset
np.random.seed(42)
X = 2 * np.random.rand(50, 1) - 1 # 50 samples, 1 feature, values between -1 and 1
# Target values with some noise, y = 0.2 + 0.1x + 0.5x^2 + noise
y = 0.2 + 0.1 * X[:, 0] + 0.5 * X[:, 0] ** 2 + np.random.randn(50) / 10

svm_poly_reg = make_pipeline(StandardScaler(),
                             SVR(kernel="poly", degree=2, C=0.01, epsilon=0.1)) # 2nd degree polynomial kernel
svm_poly_reg.fit(X, y)

In [ ]:
# extra code – this cell generates and saves Figure 5–11

svm_poly_reg2 = make_pipeline(StandardScaler(),
                             SVR(kernel="poly", degree=2, C=100))
svm_poly_reg2.fit(X, y)

fig, axes = plt.subplots(ncols=2, figsize=(9, 4), sharey=True)
plt.sca(axes[0])
plot_svm_regression(svm_poly_reg, X, y, [-1, 1, 0, 1])
plt.title(f"degree={svm_poly_reg[-1].degree}, "
          f"C={svm_poly_reg[-1].C}, "
          f"epsilon={svm_poly_reg[-1].epsilon}")
plt.ylabel("$y$", rotation=0)
plt.grid()

plt.sca(axes[1])
plot_svm_regression(svm_poly_reg2, X, y, [-1, 1, 0, 1])
plt.title(f"degree={svm_poly_reg2[-1].degree}, "
          f"C={svm_poly_reg2[-1].C}, "
          f"epsilon={svm_poly_reg2[-1].epsilon}")
plt.grid()
save_fig("svm_with_polynomial_kernel_plot")
plt.show()

---


# Under the hood
This section delves into some of the theoretical foundations of Support Vector Machines.

A linear SVM classifier's prediction is based on the decision function: $s(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b$. A prediction is positive if $s(\mathbf{x}) \ge 0$ and negative otherwise.

The decision boundary is the set of points where $s(\mathbf{x}) = 0$. The margin is the distance between the decision boundary and the closest training instances. The goal of a hard margin SVM is to maximize this margin. It can be shown that maximizing the margin is equivalent to minimizing the L2 norm of the weight vector, $\|\mathbf{w}\|$. The first figure below illustrates how a smaller weight vector $w_1$ leads to a larger margin.

For soft margin classification, SVMs use the **hinge loss** function, which penalizes instances for violating the margin. The second figure shows what this loss function looks like.

In [ ]:
# extra code – this cell generates and saves Figure 5–12

import matplotlib.patches as patches

def plot_2D_decision_function(w, b, ylabel=True, x1_lim=[-3, 3]):
    """Plots a 2D decision function for a linear SVM.

    This function visualizes the decision boundary (s = 0), the margins (s = +/- 1),
    and the decision function values (s = w * x + b) for a 1D feature space.
    It helps illustrate how the weight vector w affects the margin width.

    Args:
        w (float): The weight parameter (slope of the decision function).
        b (float): The bias parameter (intercept of the decision function).
        ylabel (bool, optional): Whether to display the y-axis label. Defaults to True.
        x1_lim (list, optional): The limits for the x-axis [min, max]. Defaults to [-3, 3].
    """
    x1 = np.linspace(x1_lim[0], x1_lim[1], 200)
    y = w * x1 + b
    half_margin = 1 / w

    plt.plot(x1, y, "b-", linewidth=2, label=r"$s = w_1 x_1$")
    plt.axhline(y=0, color='k', linewidth=1)
    plt.axvline(x=0, color='k', linewidth=1)
    rect = patches.Rectangle((-half_margin, -2), 2 * half_margin, 4,
                             edgecolor='none', facecolor='gray', alpha=0.2)
    plt.gca().add_patch(rect)
    plt.plot([-3, 3], [1, 1], "k--", linewidth=1)
    plt.plot([-3, 3], [-1, -1], "k--", linewidth=1)
    plt.plot(half_margin, 1, "k.")
    plt.plot(-half_margin, -1, "k.")
    plt.axis(x1_lim + [-2, 2])
    plt.xlabel("$x_1$")
    if ylabel:
        plt.ylabel("$s$", rotation=0, labelpad=5)
        plt.legend()
        plt.text(1.02, -1.6, "Margin", ha="left", va="center", color="k")

    plt.annotate(
        '', xy=(-half_margin, -1.6), xytext=(half_margin, -1.6),
        arrowprops={'ec': 'k', 'arrowstyle': '<->', 'linewidth': 1.5}
    )
    plt.title(f"$w_1 = {w}$")

fig, axes = plt.subplots(ncols=2, figsize=(9, 3.2), sharey=True)
plt.sca(axes[0])
plot_2D_decision_function(1, 0)
plt.grid()
plt.sca(axes[1])
plot_2D_decision_function(0.5, 0, ylabel=False)
plt.grid()
save_fig("small_w_large_margin_plot")
plt.show()

This code visualizes the relationship between the magnitude of the weight vector (`w`) and the size of the margin in a linear SVM.

### Code Explanation:

1.  **`plot_2D_decision_function(w, b, ...)`**:
    *   This function is designed to plot a simple 1D decision function `s = w * x₁ + b`.
    *   It calculates the margin width. For a linear SVM, the margin is inversely proportional to the norm of the weight vector, `||w||`. In this simple 1D case, the distance from the decision boundary (`s=0`) to the margin boundary (`s=1` or `s=-1`) is `1/w`. The total margin width is `2/w`.
    *   It plots the decision score `s` as a function of the input feature `x₁`.
    *   It draws a gray rectangle to represent the margin, whose width is `2/w`.
    *   The dashed lines at `s=1` and `s=-1` represent the margin boundaries.

2.  **Plotting**:
    *   The code creates two subplots to compare two scenarios.
    *   **Left Plot**: Calls the function with `w=1`.
    *   **Right Plot**: Calls the function with `w=0.5`.

### Meaning of the Figures:

The two plots demonstrate a core principle of SVMs: **maximizing the margin is equivalent to minimizing the norm of the weight vector `||w||`**.

*   **Left Plot (`w₁ = 1`)**:
    *   With a larger weight value, the slope of the decision function is steep.
    *   This results in a **narrow margin** (width = 2/1 = 2).

*   **Right Plot (`w₁ = 0.5`)**:
    *   With a smaller weight value, the slope is gentler.
    *   This results in a **wide margin** (width = 2/0.5 = 4).

This visualization clearly shows that to achieve a larger margin, the SVM must find a solution with a smaller `||w||`. This is why the SVM optimization objective is to minimize `½ ||w||²`.

---

Then, to visualize the **Hinge Loss** and **Squared Hinge Loss** functions, which are central to how soft margin Support Vector Machines work.

### Code Explanation:

1.  **Calculate Loss Values**:
    *   `s` represents the output of the SVM's decision function (`s = wᵀx + b`).
    *   `hinge_pos` calculates the hinge loss for a positive instance (`t=1`), which is `max(0, 1 - s)`.
    *   `hinge_neg` calculates the hinge loss for a negative instance (`t=-1`), which is `max(0, 1 + s)`.
    *   The code also calculates the squared versions of these losses.

2.  **Plotting**:
    *   It creates two subplots to compare the standard hinge loss with the squared hinge loss.
    *   In both plots, the green line shows the loss for a positive instance (`t=1`), and the red dashed line shows the loss for a negative instance (`t=-1`).

### Meaning of the Figures:

The plots show the penalty an SVM applies to a training instance based on its decision score `s`.

*   **Left Plot (Hinge Loss)**:
    *   This is the standard loss function used in `LinearSVC(loss="hinge")`.
    *   If an instance is correctly classified and lies on the correct side of the margin (i.e., `s ≥ 1` for a positive instance or `s ≤ -1` for a negative one), the loss is **zero**.
    *   If an instance violates the margin (i.e., it's inside the "street" or on the wrong side), the loss increases **linearly** with the distance from the margin boundary. This linear penalty is what gives the function its "hinge" shape.

*   **Right Plot (Squared Hinge Loss)**:
    *   This is the default loss function in `LinearSVC`.
    *   The principle is the same, but the penalty for margin violations is **quadratic**.
    *   This makes the loss function smoother and penalizes outliers (points with large margin violations) more heavily than the standard hinge loss.

To visualize the hinge loss and squared hinge loss functions, we can plot them for both positive ($t=1$) and negative ($t=-1$) target classes. The following code generates these plots, showing how the loss is zero when instances are correctly classified and outside the margin, and increases as they violate the margin.

In [ ]:
# extra code – this cell generates and saves Figure 5–13

s = np.linspace(-2.5, 2.5, 200)
hinge_pos = np.where(1 - s < 0, 0, 1 - s)  # max(0, 1 - s)
hinge_neg = np.where(1 + s < 0, 0, 1 + s)  # max(0, 1 + s)

titles = (r"Hinge loss = $max(0, 1 - s\,t)$", "Squared Hinge loss")

fix, axs = plt.subplots(1, 2, sharey=True, figsize=(8.2, 3))

for ax, loss_pos, loss_neg, title in zip(
        axs, (hinge_pos, hinge_pos ** 2), (hinge_neg, hinge_neg ** 2), titles):
    ax.plot(s, loss_pos, "g-", linewidth=2, zorder=10, label="$t=1$")
    ax.plot(s, loss_neg, "r--", linewidth=2, zorder=10, label="$t=-1$")
    ax.grid(True)
    ax.axhline(y=0, color='k')
    ax.axvline(x=0, color='k')
    ax.set_xlabel(r"$s = \mathbf{w}^\intercal \mathbf{x} + b$")
    ax.axis([-2.5, 2.5, -0.5, 2.5])
    ax.legend(loc="center right")
    ax.set_title(title)
    ax.set_yticks(np.arange(0, 2.5, 1))
    ax.set_aspect("equal")

save_fig("hinge_plot")
plt.show()

This figure illustrates the **Hinge Loss** and **Squared Hinge Loss** functions used in soft margin SVM classification.

### Mathematical Definition

Let $t$ be the target class ($t=1$ or $t=-1$) and $s = \mathbf{w}^\top \mathbf{x} + b$ be the decision function score.

1.  **Hinge Loss** (Left Plot):
    $$ J(t, s) = \max(0, 1 - t \cdot s) $$
    *   **For $t=1$ (Green line):** The loss is $\max(0, 1 - s)$. It is zero if the score $s \ge 1$ (correctly classified and safely outside the margin). If $s < 1$, the loss increases linearly.
    *   **For $t=-1$ (Red dashed line):** The loss is $\max(0, 1 + s)$. It is zero if $s \le -1$. If $s > -1$, the loss increases linearly.

2.  **Squared Hinge Loss** (Right Plot):
    $$ J(t, s) = \max(0, 1 - t \cdot s)^2 $$
    *   This is the square of the hinge loss.
    *   It penalizes outliers (large margin violations) more heavily due to the quadratic growth.
    *   It is differentiable everywhere (except at the margin boundary in the second derivative), which can simplify optimization.

### Interpretation

*   **Zero Loss**: The loss is zero only if the instance is on the correct side of the margin (i.e., $t \cdot s \ge 1$).
*   **Margin Violations**: If an instance is inside the margin ($0 < t \cdot s < 1$) or misclassified ($t \cdot s < 0$), it incurs a penalty.
*   **Linear vs. Quadratic**: The standard hinge loss applies a linear penalty proportional to the distance from the margin, while the squared hinge loss applies a quadratic penalty, making the model more sensitive to outliers.

---

In conclusion, this section provides a mathematical overview of how linear Support Vector Machines work, covering both hard and soft margin classification.

### Decision Function and Margin

For a linear SVM, the decision function for a given instance $\mathbf{x}$ is:
$s(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b$

- $\mathbf{w}$ is the weight vector.
- $b$ is the bias term.

The prediction is made based on the sign of this score: $\hat{y} = \text{sign}(s(\mathbf{x}))$. The decision boundary is the hyperplane where $s(\mathbf{x}) = 0$.

The two parallel hyperplanes that define the margins are given by $s(\mathbf{x}) = 1$ and $s(\mathbf{x}) = -1$. The distance between these two hyperplanes, known as the margin width, can be shown to be $\frac{2}{\|\mathbf{w}\|}$. To maximize this margin, we must minimize the norm of the weight vector, $\|\mathbf{w}\|$. This is equivalent to minimizing $\frac{1}{2}\|\mathbf{w}\|^2$, which is a simpler quadratic optimization problem.

### Hard Margin Classification

The objective of a hard margin SVM is to find the widest possible margin while ensuring all instances are correctly classified and outside the margin. This can be formulated as a constrained optimization problem:

**Minimize** $\frac{1}{2}\mathbf{w}^\top\mathbf{w}$
**Subject to** $t_i(\mathbf{w}^\top\mathbf{x}_i + b) \ge 1$ for $i=1, \dots, m$

Here, $t_i$ is the target label for instance $i$, defined as -1 for the negative class and +1 for the positive class. The constraint ensures that all instances are on the correct side of the margin.

### Soft Margin Classification

To handle non-linearly separable data and outliers, soft margin SVMs introduce slack variables $\zeta_i \ge 0$ for each instance. These variables measure the degree to which an instance violates the margin. The optimization problem is modified to allow for these violations, but with a penalty.

The primal optimization problem for a soft margin SVM is:

**Minimize** $\frac{1}{2}\mathbf{w}^\top\mathbf{w} + C \sum_{i=1}^m \zeta_i$
**Subject to** $t_i(\mathbf{w}^\top\mathbf{x}_i + b) \ge 1 - \zeta_i$ and $\zeta_i \ge 0$ for $i=1, \dots, m$

- The hyperparameter $C$ controls the trade-off:
    - A **small $C$** leads to a wider margin but allows more margin violations (more regularization).
    - A **large $C$** leads to a narrower margin with fewer violations (less regularization).

This formulation is equivalent to minimizing the **hinge loss**, an unconstrained objective function:

**Minimize** $\frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^m \max(0, 1 - t_i(\mathbf{w}^\top\mathbf{x}_i + b))$

The term $\max(0, 1 - t_i s(\mathbf{x}_i))$ is the hinge loss, which is zero for instances correctly classified outside the margin and increases linearly for instances that violate the margin.

---
# Extra Material

## Linear SVM classifier implementation using Batch Gradient Descent
This section provides a hands-on implementation of a linear SVM classifier from scratch using Batch Gradient Descent. The goal is to minimize the primal cost function of the soft margin SVM problem:

$J(\mathbf{w}, b) = \frac{1}{2} \mathbf{w}^\top \mathbf{w} + C \sum_{i=1}^m \max(0, 1 - t_i(\mathbf{w}^\top \mathbf{x}_i + b))$

The code defines a `MyLinearSVC` class that computes the gradients of this cost function with respect to the weights `w` and the bias `b`, and then uses these gradients to update the parameters in each training epoch. The implementation is then tested on the Iris dataset and its resulting decision boundary is compared to Scikit-Learn's `SVC` and `SGDClassifier` models.

In [ ]:
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = (iris.target == 2)

This code defines a custom linear SVM classifier class, `MyLinearSVC`, implemented from scratch using Batch Gradient Descent. It inherits from Scikit-Learn's `BaseEstimator` to be compatible with its ecosystem.

### MyLinearSVC Class Breakdown:

-   **`__init__(...)`**: The constructor initializes the hyperparameters:
    -   `C`: The regularization parameter that controls the trade-off between a wide margin and minimizing margin violations.
    -   `eta0`, `eta_d`: Parameters for the learning rate schedule. The learning rate (`eta`) decreases over time to allow for finer adjustments as training progresses.
    -   `n_epochs`: The total number of training iterations over the dataset.
    -   `random_state`: Ensures reproducible results by setting the seed for random weight initialization.

-   **`eta(self, epoch)`**: A helper method that calculates the learning rate for the current epoch based on the defined schedule.
    -   It uses the formula: $\eta = \frac{\eta_0}{1 + \eta_d \cdot \text{epoch}}$.

-   **`fit(self, X, y)`**: The core training method that implements Batch Gradient Descent.
    1.  **Initialization**: It initializes the weights `w` randomly and the bias `b` to zero. It also converts the target labels `y` (0s and 1s) into the SVM-required format `t` (-1s and 1s).
        -  `w` is initialized using a normal distribution scaled by `1/sqrt(n_features)` to ensure small initial weights.
        - `b` is initialized to zero.
        - The target labels `y` are transformed to `t` using the formula: `t = 2 * y - 1`.
    2.  **Training Loop**: It iterates for `n_epochs`. In each epoch:
        -   It identifies the **support vectors**, which are the instances that violate the margin (i.e., where $t_i(\mathbf{w}^\top \mathbf{x}_i + b) < 1$).
            - This is done using a boolean mask to filter the instances.
            -  If there are no support vectors in the current epoch, it skips the gradient update to avoid unnecessary computations.
        - The objective function is computed for monitoring purposes, although it is not used for optimization in this implementation.
            - The objective function is calculated as: $J(\mathbf{w}, b) = \frac{1}{2} \mathbf{w}^\top \mathbf{w} + C \sum_{i \in SV} (1 - t_i(\mathbf{w}^\top \mathbf{x}_i + b))$.
        -   It calculates the gradients of the cost function with respect to `w` and `b`, but **only using the support vectors**. This is a key optimization, as instances correctly classified outside the margin have a zero gradient and do not contribute to the updates.
            -  The gradient with respect to `w` is computed as: $\nabla_w = w - C \sum_{i \in SV} t_i x_i$
            -   The gradient with respect to `b` is computed as: $\nabla_b = -C \sum_{i \in SV} t_i$
        -   It updates `w` and `b` by taking a small step in the opposite direction of their respective gradients, scaled by the current learning rate.
            -  `w -= eta * grad_w`
            -  `b -= eta * grad_b`
    3.  **Store Results**: After training, it stores the final `w` and `b` as `coef_` and `intercept_` to mimic the Scikit-Learn API. It also identifies and stores the final set of support vectors.

-   **`decision_function(self, X)`**: Calculates the signed distance of the input samples `X` to the decision boundary.
    -  It computes the decision function score as: $s(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + b$.

-   **`predict(self, X)`**: Makes class predictions based on the sign of the decision function score (positive for one class, negative for the other).
    - It returns binary class labels (0 or 1) based on whether the score is non-negative or negative.

In [ ]:
from sklearn.base import BaseEstimator

class MyLinearSVC(BaseEstimator):
    def __init__(self, C=1, eta0=1, eta_d=10000, n_epochs=1000,
                 random_state=None):
        self.C = C
        self.eta0 = eta0
        self.n_epochs = n_epochs
        self.random_state = random_state
        self.eta_d = eta_d

    def eta(self, epoch):
        """Calculate the learning rate for a given epoch."""
        return self.eta0 / (epoch + self.eta_d)

    def fit(self, X, y):
        """Fit the model to the training data.

        Args:
            X (array-like): The training vectors, where n_samples is the number
            of samples and n_features is the number of features.
            y (array-like): The target values.

        Returns:
            self: Returns the instance itself.
        """
        # Random initialization
        if self.random_state:
            np.random.seed(self.random_state)
        w = np.random.randn(X.shape[1], 1)  # n feature weights
        b = 0

        # Prepare data
        t = np.array(y, dtype=np.float64).reshape(-1, 1) * 2 - 1 # convert to +1/-1
        X_t = X * t # element-wise multiplication to prepare for margin calculation
        self.Js = [] # to store objective function values

        # Training
        for epoch in range(self.n_epochs):
            support_vectors_idx = (X_t.dot(w) + t * b < 1).ravel() # indices of support vectors, where margin is violated
            X_t_sv = X_t[support_vectors_idx]
            t_sv = t[support_vectors_idx]

            # Calculate objective function value
            J = 1/2 * (w * w).sum() + self.C * ((1 - X_t_sv.dot(w)).sum() - b * t_sv.sum())
            self.Js.append(J)

            # Compute gradients
            w_gradient_vector = w - self.C * X_t_sv.sum(axis=0).reshape(-1, 1)
            b_derivative = -self.C * t_sv.sum()

            # Update parameters
            w = w - self.eta(epoch) * w_gradient_vector
            b = b - self.eta(epoch) * b_derivative


        # Store final parameters
        self.intercept_ = np.array([b])
        self.coef_ = np.array([w])
        support_vectors_idx = (X_t.dot(w) + t * b < 1).ravel()
        self.support_vectors_ = X[support_vectors_idx]
        return self

    def decision_function(self, X):
        """Predict the decision function scores for the samples in X.

        Args:
            X (array-like): The input samples.

        Returns:
            numpy.ndarray: The decision function scores.
        """
        return X.dot(self.coef_[0]) + self.intercept_[0]

    def predict(self, X):
        """Predict the class labels for the samples in X."""
        return self.decision_function(X) >= 0

This section tests the custom `MyLinearSVC` class by training it on the Iris dataset to classify Iris virginica. The training process is visualized by plotting the cost function over epochs. To validate the implementation, the resulting model's parameters and decision boundary are compared against those of Scikit-Learn's standard `SVC` and `SGDClassifier` models.

In [ ]:
C = 2
svm_clf = MyLinearSVC(C=C, eta0 = 10, eta_d = 1000, n_epochs=60000,
                      random_state=2)
svm_clf.fit(X, y)
svm_clf.predict(np.array([[5, 2], [4, 1]]))

In [ ]:
plt.plot(range(svm_clf.n_epochs), svm_clf.Js)
plt.axis((0, svm_clf.n_epochs, 0, 100))
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.grid()
plt.show()

This figure plots the value of the cost function over the training epochs for the custom `MyLinearSVC` model. The y-axis represents the loss (the value of the cost function), and the x-axis represents the number of epochs.

As you can see, the cost function steadily decreases over time and eventually flattens out. This is the expected behavior for a gradient descent-based optimizer that is successfully converging. The downward trend indicates that the algorithm is learning at each step by minimizing the cost function. The fact that the curve becomes flat towards the end suggests that the model has reached a point where further training will not significantly improve its parameters, indicating that the training has converged.

In [ ]:
print(svm_clf.intercept_, svm_clf.coef_)

### Comparing with Scikit-Learn's SVC
This code trains a standard Scikit-Learn `SVC` (Support Vector Classifier) model to serve as a benchmark for comparison against the custom `MyLinearSVC` model.

1.  **`svm_clf2 = SVC(kernel="linear", C=C)`**:
    *   An instance of the `SVC` class is created.
    *   `kernel="linear"` specifies that this should be a linear SVM, not a kernelized one. This is crucial for a fair comparison with `MyLinearSVC`.
    *   `C=C` sets the regularization hyperparameter to the same value (`C=2`) used for the custom `MyLinearSVC` model.

2.  **`svm_clf2.fit(X, y.ravel())`**:
    *   The `fit` method trains the `SVC` model on the Iris dataset (`X` and `y`).
    *   `y.ravel()` is used to ensure the target variable `y` is in the expected 1D array format.

3.  **`print(svm_clf2.intercept_, svm_clf2.coef_)`**:
    *   After training, this line prints the learned model parameters: the intercept (bias `b`) and the coefficients (weights `w`).
    *   The purpose is to compare these values directly with the parameters learned by the custom `MyLinearSVC` model (printed in the previous cell) to see how closely the from-scratch implementation matches the optimized Scikit-Learn version.

In [ ]:
svm_clf2 = SVC(kernel="linear", C=C)
svm_clf2.fit(X, y.ravel())
print(svm_clf2.intercept_, svm_clf2.coef_)

### Visualization of Decision Boundaries to Compare Models
To illustrate the performance of the custom `MyLinearSVC` model, this code visualizes its decision boundary alongside those of Scikit-Learn's `SVC` and `SGDClassifier` models.

In [ ]:
yr = y.ravel()
fig, axes = plt.subplots(ncols=2, figsize=(11, 3.2), sharey=True)
plt.sca(axes[0])
plt.plot(X[:, 0][yr==1], X[:, 1][yr==1], "g^", label="Iris virginica")
plt.plot(X[:, 0][yr==0], X[:, 1][yr==0], "bs", label="Not Iris virginica")
plot_svc_decision_boundary(svm_clf, 4, 6)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.title("MyLinearSVC")
plt.axis((4, 6, 0.8, 2.8))
plt.legend(loc="upper left")
plt.grid()

plt.sca(axes[1])
plt.plot(X[:, 0][yr==1], X[:, 1][yr==1], "g^")
plt.plot(X[:, 0][yr==0], X[:, 1][yr==0], "bs")
plot_svc_decision_boundary(svm_clf2, 4, 6)
plt.xlabel("Petal length")
plt.title("SVC")
plt.axis((4, 6, 0.8, 2.8))
plt.grid()

plt.show()

the decision boundary of our custom `MyLinearSVC` class (left) is very similar to the decision boundary of Scikit-Learn's `SVC` class (right). This demonstrates that our implementation of the linear SVM classifier using Batch Gradient Descent is working correctly.

In [ ]:
from sklearn.linear_model import SGDClassifier

sgd_clf = SGDClassifier(loss="hinge", alpha=0.017, max_iter=1000, tol=1e-3,
                        random_state=42)
sgd_clf.fit(X, y)

m = len(X)
t = np.array(y).reshape(-1, 1) * 2 - 1  # -1 if y == 0, or +1 if y == 1
X_b = np.c_[np.ones((m, 1)), X]  # Add bias input x0=1
X_b_t = X_b * t
sgd_theta = np.r_[sgd_clf.intercept_[0], sgd_clf.coef_[0]]
print(sgd_theta)
support_vectors_idx = (X_b_t.dot(sgd_theta) < 1).ravel()
sgd_clf.support_vectors_ = X[support_vectors_idx]
sgd_clf.C = C

plt.figure(figsize=(5.5, 3.2))
plt.plot(X[:, 0][yr==1], X[:, 1][yr==1], "g^")
plt.plot(X[:, 0][yr==0], X[:, 1][yr==0], "bs")
plot_svc_decision_boundary(sgd_clf, 4, 6)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.title("SGDClassifier")
plt.axis((4, 6, 0.8, 2.8))
plt.grid()

plt.show()

### Comparison with SGDClassifier

This code trains a third linear classifier, `SGDClassifier`, to demonstrate another approach to solving the same problem.

#### Code Explanation:

1.  **`SGDClassifier(...)`**: An instance of the `SGDClassifier` is created. This classifier implements regularized linear models with stochastic gradient descent (SGD) learning.
    *   `loss="hinge"`: This is the crucial parameter that makes the `SGDClassifier` behave like a linear SVM.
    *   `alpha=0.017`: This is the regularization parameter for `SGDClassifier`. It is analogous to `1/C` in `SVC` and `LinearSVC`. The value is chosen to produce a result similar to the other models.
    *   `random_state=42`: Ensures reproducibility.

2.  **Manual Support Vector Calculation**: Unlike `SVC`, `SGDClassifier` does not automatically compute and store support vectors. The code manually identifies the instances that act as support vectors (those on or inside the margin) so they can be visualized by the `plot_svc_decision_boundary` function.

3.  **Visualization**: The code then plots the data points and overlays the decision boundary learned by the `SGDClassifier`.

#### Figure Discussion:

The resulting plot shows the decision boundary for the `SGDClassifier`. When compared to the boundaries from `MyLinearSVC` and `SVC` in the previous figure, it is clear that all three models, despite using different optimization algorithms (`Batch Gradient Descent`, `liblinear`/`libsvm`, and `Stochastic Gradient Descent`), produce very similar decision boundaries. This confirms that they are all effective methods for training a linear SVM, and with proper hyperparameter tuning (`C` or `alpha`), they can converge to nearly identical models.

---


# Exercise solutions

## 1. to 8.

1. The fundamental idea behind Support Vector Machines is to fit the widest possible "street" between the classes. In other words, the goal is to have the largest possible margin between the decision boundary that separates the two classes and the training instances. When performing soft margin classification, the SVM searches for a compromise between perfectly separating the two classes and having the widest possible street (i.e., a few instances may end up on the street). Another key idea is to use kernels when training on nonlinear datasets. SVMs can also be tweaked to perform linear and nonlinear regression, as well as novelty detection.
2. After training an SVM, a _support vector_ is any instance located on the "street" (see the previous answer), including its border. The decision boundary is entirely determined by the support vectors. Any instance that is _not_ a support vector (i.e., is off the street) has no influence whatsoever; you could remove them, add more instances, or move them around, and as long as they stay off the street they won't affect the decision boundary. Computing the predictions with a kernelized SVM only involves the support vectors, not the whole training set.
3. SVMs try to fit the largest possible "street" between the classes (see the first answer), so if the training set is not scaled, the SVM will tend to neglect small features (see Figure 5–2).
4. You can use the `decision_function()` method to get confidence scores. These scores represent the distance between the instance and the decision boundary. However, they cannot be directly converted into an estimation of the class probability. If you set `probability=True` when creating an `SVC`, then at the end of training it will use 5-fold cross-validation to generate out-of-sample scores for the training samples, and it will train a `LogisticRegression` model to map these scores to estimated probabilities. The `predict_proba()` and `predict_log_proba()` methods will then be available.
5. All three classes can be used for large-margin linear classification. The `SVC` class also supports the kernel trick, which makes it capable of handling nonlinear tasks. However, this comes at a cost: the `SVC` class does not scale well to datasets with many instances. It does scale well to a large number of features, though. The `LinearSVC` class implements an optimized algorithm for linear SVMs, while `SGDClassifier` uses Stochastic Gradient Descent. Depending on the dataset `LinearSVC` may be a bit faster than `SGDClassifier`, but not always, and `SGDClassifier` is more flexible, plus it supports incremental learning.
6. If an SVM classifier trained with an RBF kernel underfits the training set, there might be too much regularization. To decrease it, you need to increase `gamma` or `C` (or both).
7. A Regression SVM model tries to fit as many instances within a small margin around its predictions. If you add instances within this margin, the model will not be affected at all: it is said to be _ϵ-insensitive_.
8. The kernel trick is mathematical technique that makes it possible to train a nonlinear SVM model. The resulting model is equivalent to mapping the inputs to another space using a nonlinear transformation, then training a linear SVM on the resulting high-dimensional inputs. The kernel trick gives the same result without having to transform the inputs at all.

---
# 9.

_Exercise: Train a `LinearSVC` on a linearly separable dataset. Then train an `SVC` and a `SGDClassifier` on the same dataset. See if you can get them to produce roughly the same model._

Let's use the Iris dataset: the Iris Setosa and Iris Versicolor classes are linearly separable.

In [ ]:
from sklearn import datasets

iris = datasets.load_iris(as_frame=True)
X = iris.data[["petal length (cm)", "petal width (cm)"]].values
y = iris.target

setosa_or_versicolor = (y == 0) | (y == 1)
X = X[setosa_or_versicolor]
y = y[setosa_or_versicolor]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6, 4))
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", label="Iris setosa")
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper left")
plt.grid()
plt.show()

Now let's build and train 3 models:
* Remember that `LinearSVC` uses `loss="squared_hinge"` by default, so if we want all 3 models to produce similar results, we need to set `loss="hinge"`.
* Also, the `SVC` class uses an RBF kernel by default, so we need to set `kernel="linear"` to get similar results as the other two models.
* Lastly, the `SGDClassifier` class does not have a `C` hyperparameter, but it has another regularization hyperparameter called `alpha`, so we can tweak it to get similar results as the other two models.

In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

C = 5
alpha = 0.05

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lin_clf = LinearSVC(loss="hinge", C=C, dual=True, random_state=42).fit(X_scaled, y)
svc_clf = SVC(kernel="linear", C=C).fit(X_scaled, y)
sgd_clf = SGDClassifier(alpha=alpha, random_state=42).fit(X_scaled, y)

In [ ]:
lin_clf.coef_, lin_clf.intercept_

Let's plot the decision boundaries of these three models:

To calculate the decision boundary for a linear SVM (or any linear classifier), you need to find the line where the decision function equals 0.

The decision function is given by:

$w_0 \cdot x_0 + w_1 \cdot x_1 + b = 0$

Where:
*   $w_0$ and $w_1$ are the weights (coefficients) for the two features.
*   $b$ is the bias (intercept).
*   $x_0$ is the first feature (e.g., petal length).
*   $x_1$ is the second feature (e.g., petal width).

Solving for $x_1$ (to plot it as $y$ on a 2D graph):

$x_1 = - \frac{w_0}{w_1} x_0 - \frac{b}{w_1}$

Since your model was trained on scaled data, the coefficients correspond to the scaled features. To plot the boundary on the original scale, you calculate points in the scaled space and then use `scaler.inverse_transform()` to convert them back.


In [ ]:
def compute_decision_boundary(model):
    """Computes the decision boundary for a linear classifier.

    Args:
        model: The trained linear classifier model.

    Returns:
        numpy.ndarray: Two points defining the decision boundary line in the
            original feature space.
    """
    w = -model.coef_[0, 0] / model.coef_[0, 1]
    b = -model.intercept_[0] / model.coef_[0, 1]
    return scaler.inverse_transform([[-10, -10 * w + b], [10, 10 * w + b]])

lin_line = compute_decision_boundary(lin_clf)
svc_line = compute_decision_boundary(svc_clf)
sgd_line = compute_decision_boundary(sgd_clf)

# Plot all three decision boundaries
plt.figure(figsize=(11, 4))
plt.plot(lin_line[:, 0], lin_line[:, 1], "k:", label="LinearSVC")
plt.plot(svc_line[:, 0], svc_line[:, 1], "b--", linewidth=2, label="SVC")
plt.plot(sgd_line[:, 0], sgd_line[:, 1], "r-", label="SGDClassifier")
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs") # label="Iris versicolor"
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo") # label="Iris setosa"
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper center")
plt.axis((0, 5.5, 0, 2))
plt.grid()

plt.show()

Close enough!

In [ ]:
def compute_decision_boundary_and_margin(model, scaler, x_range=(-10, 10)):
    """Computes the decision boundary and margin boundaries for a linear classifier.

    Args:
        model: The trained linear classifier model with coef_ and intercept_ attributes.
        scaler: The StandardScaler used to transform the training data.
        x_range (tuple): The range of x values for computing the boundary points.

    Returns:
        tuple: Three numpy arrays (decision_boundary, upper_margin, lower_margin)
            each containing two points defining a line in the original feature space,
            and the scalar margin width (1 / ||w||).
    """
    w = model.coef_[0]  # weight vector [w0, w1]
    b = model.intercept_[0]  # bias term
    
    # Decision boundary: w0*x0 + w1*x1 + b = 0 => x1 = -(w0/w1)*x0 - b/w1
    slope = -w[0] / w[1]
    intercept = -b / w[1]
    
    # Margin: distance from decision boundary = 1/||w||
    margin_width_scaled = 1 / np.linalg.norm(w)
    
    # Compute three parallel lines in scaled space
    x_scaled = np.array(x_range)
    decision_y = slope * x_scaled + intercept
    upper_margin_y = decision_y + margin_width_scaled
    lower_margin_y = decision_y - margin_width_scaled
    
    # Transform back to original feature space
    decision_boundary = scaler.inverse_transform(np.column_stack([x_scaled, decision_y]))
    upper_margin = scaler.inverse_transform(np.column_stack([x_scaled, upper_margin_y]))
    lower_margin = scaler.inverse_transform(np.column_stack([x_scaled, lower_margin_y]))

    # Calculate margin width in original feature space
    w_unscaled = w / scaler.scale_
    margin_width = 1 / np.linalg.norm(w_unscaled)

    return decision_boundary, upper_margin, lower_margin, margin_width


def plot_classifier_with_margins(model, scaler, X, y, color, label, linestyle='-'):
    """Plots a classifier's decision boundary and margins.

    Args:
        model: The trained linear classifier model.
        scaler: The StandardScaler used to transform the training data.
        X (numpy.ndarray): The original (unscaled) feature data.
        y (numpy.ndarray): The target labels.
        color (str): The color for the decision boundary.
        label (str): The label for the legend.
        linestyle (str): The line style for the decision boundary.
    """
    decision_line, upper_margin, lower_margin, margin_width = compute_decision_boundary_and_margin(model, scaler)
    
    # Plot decision boundary
    plt.plot(decision_line[:, 0], decision_line[:, 1], 
             color=color, linestyle=linestyle, linewidth=2,
             label=f"{label} (margin={margin_width:.3f})")
    
    # Plot margins (dashed, lighter color)
    plt.plot(upper_margin[:, 0], upper_margin[:, 1], 
             color=color, linestyle='-.', linewidth=1, alpha=0.6)
    plt.plot(lower_margin[:, 0], lower_margin[:, 1], 
             color=color, linestyle='-.', linewidth=1, alpha=0.6)


# Create the comparison plot
plt.figure(figsize=(11, 4))

# Plot decision boundaries and margins for each classifier
plot_classifier_with_margins(lin_clf, scaler, X, y, 'black', 'LinearSVC', linestyle='-')
plot_classifier_with_margins(svc_clf, scaler, X, y, 'blue', 'SVC', linestyle='-')
plot_classifier_with_margins(sgd_clf, scaler, X, y, 'red', 'SGDClassifier', linestyle='-')

# Plot data points
plt.plot(X[:, 0][y==1], X[:, 1][y==1], "bs", label="Iris versicolor")
plt.plot(X[:, 0][y==0], X[:, 1][y==0], "yo", label="Iris setosa")

# Format plot
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.legend(loc="upper center")
plt.axis((0, 5.5, 0, 2))
plt.grid()
plt.title("Linear Classifier Comparison with Margins")

plt.show()

---
# 10.

_Exercise: Train an SVM classifier on the Wine dataset, which you can load using `sklearn.datasets.load_wine()`. This dataset contains the chemical analysis of 178 wine samples produced by 3 different cultivators: the goal is to train a classification model capable of predicting the cultivator based on the wine's chemical analysis. Since SVM classifiers are binary classifiers, you will need to use one-versus-all to classify all 3 classes. What accuracy can you reach?_

First, let's fetch the dataset, look at its description, then split it into a training set and a test set:

In [114]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)

In [115]:
print(wine.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

In [116]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    wine.data, wine.target, random_state=42)

print(f"Length of training set is {len(X_train)}")

Length of training set is 133


In [117]:
X_train.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
100,12.08,2.08,1.70,17.5,97.0,2.23,2.17,0.26,1.40,3.30,1.27,2.96,710.0
122,12.42,4.43,2.73,26.5,102.0,2.20,2.13,0.43,1.71,2.08,0.92,3.12,365.0
154,12.58,1.29,2.10,20.0,103.0,1.48,0.58,0.53,1.40,7.60,0.58,1.55,640.0
51,13.83,1.65,2.60,17.2,94.0,2.45,2.99,0.22,2.29,5.60,1.24,3.37,1265.0


In [118]:
y_train.head()

,target
2,0
100,1
122,1
154,2
51,0


Let's start simple, with a linear SVM classifier. It will automatically use the One-vs-All (also called One-vs-the-Rest, OvR) strategy, so there's nothing special we need to do to handle multiple classes. Easy, right?

In [119]:
lin_clf = LinearSVC(dual=True, random_state=42)
lin_clf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC(dual=True, random_state=42)

Oh no! It failed to converge. Can you guess why? Do you think we must just increase the number of training iterations? Let's see:

In [120]:
lin_clf = LinearSVC(max_iter=1_000_000, dual=True, random_state=42)
lin_clf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC(dual=True, max_iter=1000000, random_state=42)

Even with one million iterations, it still did not converge. There must be another problem.

Let's still evaluate this model with `cross_val_score`, it will serve as a baseline:

In [121]:
from sklearn.model_selection import cross_val_score

cross_val_score(lin_clf, X_train, y_train).mean()

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  w

np.float64(0.90997150997151)

Well 91% accuracy on this dataset is not great. So did you guess what the problem is?

That's right, we forgot to scale the features! Always remember to scale the features when using SVMs:

In [122]:
from sklearn.pipeline import make_pipeline

lin_clf = make_pipeline(StandardScaler(),
                        LinearSVC(dual=True, random_state=42))
lin_clf.fit(X_train, y_train)


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearsvc', LinearSVC(dual=True, random_state=42))])

Now it converges without any problem. Let's measure its performance:

In [123]:
from sklearn.model_selection import cross_val_score

cross_val_score(lin_clf, X_train, y_train, cv=3).mean()

np.float64(0.9774410774410773)

Nice! We get 97.7% accuracy, that's much better.

Let's see if a kernelized SVM will do better. We will use a default `SVC` for now:

In [124]:
svm_clf = make_pipeline(StandardScaler(), SVC(random_state=42))
cross_val_score(svm_clf, X_train, y_train).mean()

np.float64(0.9698005698005698)

That's not better, but perhaps we need to do a bit of hyperparameter tuning:

In [125]:
svm_clf[-1].__dict__.keys()

dict_keys(['decision_function_shape', 'break_ties', 'kernel', 'degree', 'gamma', 'coef0', 'tol', 'C', 'nu', 'epsilon', 'shrinking', 'probability', 'cache_size', 'class_weight', 'verbose', 'max_iter', 'random_state'])

In [126]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform

param_distrib = {
    "svc__gamma": loguniform(0.001, 0.1),
    "svc__C": uniform(1, 20),
    "svc__kernel": ["rbf", "poly", "sigmoid"],
    "svc__degree": range(1,10)
}
rnd_search_cv = RandomizedSearchCV(svm_clf, param_distrib, n_iter=100, cv=5,
                                   random_state=42, verbose=3)
rnd_search_cv.fit(X_train, y_train)
rnd_search_cv.best_estimator_

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5] END svc__C=8.49080237694725, svc__degree=8, svc__gamma=0.015751320499779727, svc__kernel=sigmoid;, score=0.963 total time=   0.0s
[CV 2/5] END svc__C=8.49080237694725, svc__degree=8, svc__gamma=0.015751320499779727, svc__kernel=sigmoid;, score=0.963 total time=   0.0s
[CV 3/5] END svc__C=8.49080237694725, svc__degree=8, svc__gamma=0.015751320499779727, svc__kernel=sigmoid;, score=0.963 total time=   0.0s
[CV 4/5] END svc__C=8.49080237694725, svc__degree=8, svc__gamma=0.015751320499779727, svc__kernel=sigmoid;, score=0.962 total time=   0.0s
[CV 5/5] END svc__C=8.49080237694725, svc__degree=8, svc__gamma=0.015751320499779727, svc__kernel=sigmoid;, score=1.000 total time=   0.0s
[CV 1/5] END svc__C=9.916655057071823, svc__degree=7, svc__gamma=0.0013066739238053278, svc__kernel=rbf;, score=0.963 total time=   0.0s
[CV 2/5] END svc__C=9.916655057071823, svc__degree=7, svc__gamma=0.0013066739238053278, svc__kernel=rbf;,

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc',
                 SVC(C=np.float64(11.214946051551316), degree=9,
                     gamma=np.float64(0.009325238385910697),
                     random_state=42))])

In [127]:
print(f'The best estimator is {rnd_search_cv.best_estimator_[-1]}')
print(f'The best kernel is {rnd_search_cv.best_estimator_[-1].kernel}')
print(f'The best score is {rnd_search_cv.best_score_:.6f}')


The best estimator is SVC(C=np.float64(11.214946051551316), degree=9,
    gamma=np.float64(0.009325238385910697), random_state=42)
The best kernel is rbf
The best score is 0.992593


Ah, this looks excellent! Let's select this model. Now we can test it on the test set:

In [128]:
rnd_search_cv.score(X_test, y_test)

0.9777777777777777

This tuned kernelized SVM performs better than the `LinearSVC` model, but we get a lower score on the test set than we measured using cross-validation. This is quite common: since we did so much hyperparameter tuning, we ended up slightly overfitting the cross-validation test sets. It's tempting to tweak the hyperparameters a bit more until we get a better result on the test set, but this would probably not help, as we would just start overfitting the test set. Anyway, this score is not bad at all, so let's stop here.

---
# 11.

_Exercise: Train and fine-tune an SVM regressor on the California housing dataset. You can use the original dataset rather than the tweaked version we used in Chapter 2. The original dataset can be fetched using `sklearn.datasets.fetch_california_housing()`. The targets represent hundreds of thousands of dollars. Since there are over 20,000 instances, SVMs can be slow, so for hyperparameter tuning you should use much less instances (e.g., 2,000), to test many more hyperparameter combinations. What is your best model's RMSE?_

Let's load the dataset:

In [1]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
X = housing.data
y = housing.target

In [2]:
print(housing.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

Split it into a training set and a test set:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [4]:
print(f'Size of the test set: {len(X_test)}')

Size of the test set: 2064


Don't forget to scale the data!

Let's train a simple `LinearSVR` first:

In [5]:
from sklearn.svm import LinearSVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

lin_svr = make_pipeline(StandardScaler(), LinearSVR(dual=True, random_state=42))
lin_svr.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearsvr', LinearSVR(dual=True, random_state=42))])

In [6]:
print(f'Attributes of the LinearSVR model:')
for key in lin_svr[-1].__dict__.keys():
    print(f'{key}: {lin_svr[-1].__dict__[key]}')


Attributes of the LinearSVR model:
tol: 0.0001
C: 1.0
epsilon: 0.0
fit_intercept: True
intercept_scaling: 1.0
verbose: 0
random_state: 42
max_iter: 1000
dual: True
loss: epsilon_insensitive
n_features_in_: 8
coef_: [ 0.9079411   0.09383611 -0.38518795  0.45931055  0.0137142  -0.80180999
 -0.75063195 -0.74522607]
intercept_: [1.93610818]
n_iter_: 1000


In [ ]:
print(f"Number of iterations used by LinearSVR: {lin_svr[-1].n_iter_}")

Number of iterations used by LinearSVR: 1000


It did not converge, so let's increase `max_iter`:

In [8]:
lin_svr = make_pipeline(StandardScaler(),
                        LinearSVR(max_iter=5000, dual=True, random_state=42))
lin_svr.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearsvr',
                 LinearSVR(dual=True, max_iter=5000, random_state=42))])

Let's see how it performs on the training set:

**Warning**: In recent versions of Scikit-Learn, you must use `root_mean_squared_error()` to compute the RMSE, instead of `mean_squared_error(labels, predictions, squared=False)`. The following `try`/`except` block tries to import `root_mean_squared_error`, and if it fails it just defines it.

In [ ]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    print("Defining root_mean_squared_error function as it is not available in this version of sklearn.")
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

In [10]:
y_pred = lin_svr.predict(X_train)
rmse = root_mean_squared_error(y_train, y_pred)
print(f'{rmse} is the RMSE on the training set')

1.057246989905268 is the RMSE on the training set


In this dataset, the targets represent hundreds of thousands of dollars. The RMSE gives a rough idea of the kind of error you should expect (with a higher weight for large errors): so with this model we can expect errors close to $105,725! Not great. Let's see if we can do better with an RBF Kernel. We will use randomized search with cross validation to find the appropriate hyperparameter values for `C` and `gamma`:

In [11]:
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform

svm_reg = make_pipeline(StandardScaler(), SVR())

param_distrib = {
    "svr__gamma": loguniform(0.001, 0.1),
    "svr__C": uniform(1, 10)
}
rnd_search_cv = RandomizedSearchCV(svm_reg, param_distrib,
                                   n_iter=100, cv=3, random_state=42, verbose=3)
rnd_search_cv.fit(X_train, y_train)

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('standardscaler',
                                              StandardScaler()),
                                             ('svr', SVR())]),
                   n_iter=100,
                   param_distributions={'svr__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78866bbdc260>,
                                        'svr__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x78867017c050>},
                   random_state=42)

This code performs hyperparameter tuning for a Support Vector Regressor (SVR) with an RBF kernel on the California housing dataset. It uses `RandomizedSearchCV` to efficiently explore a range of hyperparameters for `gamma` (kernel coefficient) and `C` (regularization parameter), aiming to find the best combination that minimizes the cross-validation error. The pipeline includes `StandardScaler` to normalize the features, which is crucial for SVM performance. With 100 iterations and 3-fold cross-validation, this randomized search balances exploration and computational efficiency on the training set. After fitting, the best estimator can be accessed via `rnd_search_cv.best_estimator_`, and its performance evaluated on the test set to assess generalization. This approach helps optimize the SVR for better RMSE on the housing price prediction task.

In [12]:
rnd_search_cv.best_estimator_

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svr',
                 SVR(C=np.float64(4.63629602379294),
                     gamma=np.float64(0.08781408196485979)))])

In [16]:
from sklearn.model_selection import cross_val_score
-cross_val_score(rnd_search_cv.best_estimator_, X_train, y_train,
                 scoring="neg_root_mean_squared_error")

array([0.57256491, 0.59928204, 0.57629751, 0.56502599, 0.59432526])

Looks much better than the linear model. Let's select this model and evaluate it on the test set:

In [17]:
y_pred = rnd_search_cv.best_estimator_.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
print(f'{rmse} is the RMSE on the test set')

0.592432637319211 is the RMSE on the test set


So SVMs worked very well on the Wine dataset, but not so much on the California Housing dataset. In Chapter 2, we found that Random Forests worked better for that dataset.

And that's all for today!